In [ ]:
import json
from cyclonedx.validation.json import JsonValidator, JsonStrictValidator
from cyclonedx.schema import SchemaVersion

import logging
import sys
from cyclonedx.model.bom import Bom
# from cyclonedx.model.json import json_loads
# from cyclonedx.exceptions import MissingOptionalDependencyException

# def validate_cyclonedx_bom(payload, strict=False):
#     """
#     Validates a CycloneDX BOM JSON payload using the CycloneDX Python Library.
    
#     Args:
#         payload (str | dict): The JSON payload as a string or a Python dictionary.
#         strict (bool): If True, use the strict JSON validator. Default is False (standard validation).
    
#     Returns:
#         dict: Result dictionary with success status and message.
#     """
#     try:
#         # If the payload is a string, parse it as JSON.
#         if isinstance(payload, str):
#             json_data = payload
#         else:
#             json_data = json.dumps(payload)

#         # Select the validator based on the 'strict' flag.
#         if strict:
#             validator = JsonStrictValidator(schema_version=SchemaVersion.V1_4)
#         else:
#             validator = JsonValidator(schema_version=SchemaVersion.V1_4)
#             # print(dir(validator))

#         # Use validate_str() method which returns validation errors
#         validation_errors = validator.validate_str(json_data)

#         if not validation_errors:
#             return {"success": True, "message": "Validation passed."}
#         else:
#             return {"success": False, "message": f"Validation failed with errors: {validation_errors}"}

#     except json.JSONDecodeError:
#         return {"success": False, "message": "Invalid JSON format in the payload."}
#     except Exception as e:
#         return {"success": False, "message": f"An error occurred: {str(e)}"}
    
# def validate_cyclone(cyclone_content):
#     """
#     Validates a CycloneDX payload.

#     Args:
#         cyclone_content (dict): The parsed JSON content of the CycloneDX payload.

#     Returns:
#         bool: True if the payload is valid, False otherwise.
#     """
#     # logger = logging.getLogger(__name__)
    
#     # Check if the payload has the required fields
#     if "bomFormat" in cyclone_content and "specVersion" in cyclone_content and "serialNumber" in cyclone_content:
#         # logger.info("Basic CycloneDX format validation passed")
#         return True

#     my_json_validator = JsonStrictValidator(SchemaVersion.V1_4)
#     try:
#         json_data = json.dumps(cyclone_content)
#         validation_errors = my_json_validator.validate_str(json_data)
#         if validation_errors:
#             # logger.info('JSON invalid\nValidationError: %s', repr(validation_errors))
#             sys.exit(2)
#         logger.info('JSON valid')
#     except MissingOptionalDependencyException as error:
#         logger.info('JSON-validation was skipped due to %s', error)
#     bom_from_json = Bom.from_json(json_loads(json_data))
#     logger.info('bom_from_json %s', repr(bom_from_json))

#     return False

def validate_cyclonedx_smart(payload, strict=False, thorough=True):
    """
    Smart validation of a CycloneDX BOM that combines quick field checking and schema validation.
    Automatically detects the schema version from the payload.
    
    Args:
        payload (str | dict): The JSON payload as a string or a Python dictionary.
        strict (bool): If True, use the strict JSON validator. Default is False.
        thorough (bool): If True, performs complete schema validation. If False, only checks required fields.
    
    Returns:
        dict: Result dictionary with success status and message.
    """
    try:
        # Parse the payload if it's a string
        if isinstance(payload, str):
            try:
                content = json.loads(payload)
                json_data = payload  # Keep original string for validation
            except json.JSONDecodeError:
                return {"success": False, "message": "Invalid JSON format in the payload."}
        else:
            content = payload
            json_data = json.dumps(payload)
            
        # Quick check for required fields
        if "bomFormat" not in content or "specVersion" not in content or "serialNumber" not in content:
            return {"success": False, "message": "Missing required fields: bomFormat, specVersion, or serialNumber"}
        
        # Get the schema version from the payload
        spec_version = content.get("specVersion", "1.4")  # Default to 1.4 if not specified
        
        # Map the specVersion to a SchemaVersion enum
        schema_version_map = {
            "1.0": SchemaVersion.V1_0,
            "1.1": SchemaVersion.V1_1,
            "1.2": SchemaVersion.V1_2,
            "1.3": SchemaVersion.V1_3,
            "1.4": SchemaVersion.V1_4,
            "1.5": SchemaVersion.V1_5,
            "1.6": SchemaVersion.V1_6
        }
        
        schema_version = schema_version_map.get(spec_version, SchemaVersion.V1_4)
        
        # If only basic validation is requested, return success
        if not thorough:
            return {"success": True, "message": f"Basic validation passed. Detected schema version: {spec_version}"}
        
        # Perform thorough schema validation
        validator = JsonStrictValidator(schema_version=schema_version) if strict else JsonValidator(schema_version=schema_version)
        
        validation_errors = validator.validate_str(json_data)
        
        if not validation_errors:
            return {
                "success": True, 
                "message": f"Validation passed. Schema version: {spec_version}",
                "schema_version": spec_version
            }
        else:
            return {
                "success": False, 
                "message": f"Validation failed with errors: {validation_errors}",
                "schema_version": spec_version
            }
            
    except Exception as e:
        return {"success": False, "message": f"An error occurred: {str(e)}"}


payload_correct_bom_master1 = {
  "bomFormat": "CycloneDX",
  "specVersion": "1.6",
  "serialNumber": "urn:uuid:e8c355aa-2142-4084-a8c7-6d42c8610ba2",
  "version": 1,
  "metadata": {
    "timestamp": "2024-01-09T12:00:00Z",
    "component": {
      "type": "application",
      "name": "my application",
      "version": "1.0"
    }
  },
  "components": [
    {
      "name": "google.com",
      "type": "cryptographic-asset",
      "bom-ref": "crypto/certificate/google.com@sha256:1e15e0fbd3ce95bde5945633ae96add551341b11e5bae7bba12e98ad84a5beb4",
      "cryptoProperties": {
        "assetType": "certificate",
        "certificateProperties": {
          "subjectName": "CN = www.google.com",
          "issuerName": "C = US, O = Google Trust Services LLC, CN = GTS CA 1C3",
          "notValidBefore": "2016-11-21T08:00:00Z",
          "notValidAfter": "2017-11-22T07:59:59Z",
          "signatureAlgorithmRef": "crypto/algorithm/sha-512-rsa@1.2.840.113549.1.1.13",
          "subjectPublicKeyRef": "crypto/key/rsa-2048@1.2.840.113549.1.1.1",
          "certificateFormat": "X.509",
          "certificateExtension": "crt"
        }
      }
    },
    {
      "name": "SHA512withRSA",
      "type": "cryptographic-asset",
      "bom-ref": "crypto/algorithm/sha-512-rsa@1.2.840.113549.1.1.13",
      "cryptoProperties": {
        "assetType": "algorithm",
        "algorithmProperties": {
          "parameterSetIdentifier": "512",
          "executionEnvironment": "software-plain-ram",
          "implementationPlatform": "x86_64",
          "certificationLevel": [ "none" ],
          "cryptoFunctions": [ "digest" ],
          "nistQuantumSecurityLevel": 0
        },
        "oid": "1.2.840.113549.1.1.13"
      }
    },
    {
      "name": "RSA-2048",
      "type": "cryptographic-asset",
      "bom-ref": "crypto/key/rsa-2048@1.2.840.113549.1.1.1",
      "cryptoProperties": {
        "assetType": "related-crypto-material",
        "relatedCryptoMaterialProperties": {
          "type": "public-key",
          "id": "2e9ef09e-dfac-4526-96b4-d02f31af1b22",
          "state": "active",
          "size": 2048,
          "algorithmRef": "crypto/algorithm/rsa-2048@1.2.840.113549.1.1.1",
          "securedBy": {
            "mechanism": "None"
          },
          "creationDate": "2016-11-21T08:00:00Z",
          "activationDate": "2016-11-21T08:20:00Z"
        },
        "oid": "1.2.840.113549.1.1.1"
      }
    },
    {
      "name": "RSA-2048",
      "type": "cryptographic-asset",
      "bom-ref": "crypto/algorithm/rsa-2048@1.2.840.113549.1.1.1",
      "cryptoProperties": {
        "assetType": "algorithm",
        "algorithmProperties": {
          "parameterSetIdentifier": "2048",
          "executionEnvironment": "software-plain-ram",
          "implementationPlatform": "x86_64",
          "certificationLevel": [ "none" ],
          "cryptoFunctions": [ "encapsulate", "decapsulate" ]
        },
        "oid": "1.2.840.113549.1.1.1"
      }
    }
  ]
}

payload_correct_bom_master2 = {
  "bomFormat": "CycloneDX",
  "specVersion": "1.4",
  "serialNumber": "urn:uuid:3e671687-395b-41f5-a30f-a58921a69b79",
  "version": 1,
  "metadata": {
    "timestamp": "2021-01-10T12:00:00Z",
    "component": {
      "bom-ref": "acme-application",
      "type": "application",
      "name": "Acme Cloud Example",
      "version": "2022-1"
    }
  },
  "services": [
    {
      "bom-ref": "api-gateway",
      "provider": {
        "name": "Acme Inc",
        "url": [ "https://example.com" ]
      },
      "group": "com.example",
      "name": "API Gateway",
      "version": "2022-1",
      "description": "Example API Gateway",
      "endpoints": [
        "https://example.com/",
        "https://example.com/app"
      ],
      "authenticated": False,
      "x-trust-boundary": True,
      "data": [
        {
          "classification": "PII",
          "flow": "bi-directional"
        },
        {
          "classification": "PIFI",
          "flow": "bi-directional"
        },
        {
          "classification": "Public",
          "flow": "bi-directional"
        }
      ],
      "externalReferences": [
        {
          "type": "documentation",
          "url": "http://example.com/app/swagger"
        }
      ],
      "services": [
        {
          "bom-ref": "ms-1.example.com",
          "provider": {
            "name": "Acme Inc",
            "url": [ "https://example.com" ]
          },
          "group": "com.example",
          "name": "Microservice 1",
          "version": "2022-1",
          "description": "Example Microservice",
          "endpoints": [
            "https://ms-1.example.com"
          ],
          "authenticated":True,
          "x-trust-boundary": False,
          "data": [
            {
              "classification": "PII",
              "flow": "bi-directional"
            }
          ],
          "externalReferences": [
            {
              "type": "documentation",
              "url": "https://ms-1.example.com/swagger"
            }
          ]
        },
        {
          "bom-ref": "ms-2.example.com",
          "provider": {
            "name": "Acme Inc",
            "url": [ "https://example.com" ]
          },
          "group": "com.example",
          "name": "Microservice 2",
          "version": "2022-1",
          "description": "Example Microservice",
          "endpoints": [
            "https://ms-2.example.com"
          ],
          "authenticated":True,
          "x-trust-boundary": False,
          "data": [
            {
              "classification": "PIFI",
              "flow": "bi-directional"
            }
          ],
          "externalReferences": [
            {
              "type": "documentation",
              "url": "https://ms-2.example.com/swagger"
            }
          ]
        },
        {
          "bom-ref": "ms-3.example.com",
          "provider": {
            "name": "Acme Inc",
            "url": [ "https://example.com" ]
          },
          "group": "com.example",
          "name": "Microservice 3",
          "version": "2022-1",
          "description": "Example Microservice",
          "endpoints": [
            "https://ms-3.example.com"
          ],
          "authenticated":True,
          "x-trust-boundary": False,
          "data": [
            {
              "classification": "Public",
              "flow": "bi-directional"
            }
          ],
          "externalReferences": [
            {
              "type": "documentation",
              "url": "https://ms-3.example.com/swagger"
            }
          ]
        },
        {
          "bom-ref": "ms-1-pgsql.example.com",
          "group": "org.postgresql",
          "name": "Postgres",
          "version": "14.1",
          "description": "Postgres database for Microservice #1",
          "endpoints": [
            "https://ms-1-pgsql.example.com:5432"
          ],
          "authenticated":True,
          "x-trust-boundary": False,
          "data": [
            {
              "classification": "PII",
              "flow": "bi-directional"
            }
          ]
        },
        {
          "bom-ref": "s3-example.amazon.com",
          "group": "com.amazon",
          "name": "S3",
          "description": "S3 bucket",
          "endpoints": [
            "https://s3-example.amazon.com"
          ],
          "authenticated":True,
          "x-trust-boundary":True,
          "data": [
            {
              "classification": "Public",
              "flow": "bi-directional"
            }
          ]
        }
      ]
    }
  ],
  "dependencies": [
    {
      "ref": "acme-application",
      "dependsOn": [ "api-gateway" ]
    },
    {
      "ref": "api-gateway",
      "dependsOn": [
        "ms-1.example.com",
        "ms-2.example.com",
        "ms-3.example.com"
      ]
    },
    {
      "ref": "ms-1.example.com",
      "dependsOn": [ "ms-1-pgsql.example.com" ]
    },
    {
      "ref": "ms-2.example.com",
      "dependsOn": [ ]
    },
    {
      "ref": "ms-3.example.com",
      "dependsOn": [ "s3-example.amazon.com" ]
    }
  ]
}


payload_correct_CCM1 = {
    "bomFormat": "CycloneDX",
    "specVersion": "1.6",
    "serialNumber": "urn:uuid:9b553688-c976-4d0a-9d22-4e3b59d174e2",
    "version": 1,
    "metadata": {
        "timestamp": "2025-02-25T10:32:41Z",
        "component": {
            "type": "application",
            "name": "my application",
            "version": "1.0"
        }
    },
    "components": [
        {
            "type": "certificate",
            "name": "MailHog TLS Certificate",
            "version": "1.0",
            "description": "Self-signed TLS certificate for MailHog",
            "licenses": [],
            "externalReferences": [
                {
                    "type": "distribution",
                    "url": "file:/home/localadmin/mailhog.crt"
                }
            ]
        }
    ]
}

payload_correct_CCM2 = {
  "bomFormat": "CycloneDX",
  "specVersion": "1.4",
  "serialNumber": "urn:uuid:123e4567-e89b-12d3-a456-426614174002",
  "version": 1,
  "metadata": {
    "timestamp": "2025-02-25T10:51:08Z",
    "tools": [
      {
        "vendor": "cyclonedx",
        "name": "cyclonedx-python",
        "version": "1.0"
      }
    ],
    "component": {
      "type": "application",
      "name": "GymManagementSystem",
      "version": "latest",
      "description": "A Python-Flask based web application for managing gym operations, utilizing a MySQL database.",
      "licenses": [],
      "purl": "pkg:github/EswarGitHub/GymManagementSystem"
    },
    "authors": [
      {
        "name": "Eswar",
        "email": "eswar@example.com"
      }
    ]
  },
  "components": [
    {
      "type": "library",
      "name": "Flask",
      "version": "1.1.2",
      "description": "A micro web framework written in Python.",
      "licenses": [],
      "purl": "pkg:pypi/Flask@1.1.2"
    },
    {
      "type": "library",
      "name": "Flask-MySQL",
      "version": "1.5.1",
      "description": "Flask extension that provides MySQL support.",
      "licenses": [],
      "purl": "pkg:pypi/Flask-MySQL@1.5.1"
    },
    {
      "type": "library",
      "name": "Werkzeug",
      "version": "1.0.1",
      "description": "A comprehensive WSGI web application library.",
      "licenses": [],
      "purl": "pkg:pypi/Werkzeug@1.0.1"
    },
    {
      "type": "library",
      "name": "Jinja2",
      "version": "2.11.3",
      "description": "A templating engine for Python.",
      "licenses": [],
      "purl": "pkg:pypi/Jinja2@2.11.3"
    },
    {
      "type": "library",
      "name": "MarkupSafe",
      "version": "1.1.1",
      "description": "Implements a XML/HTML/XHTML Markup safe string for Python.",
      "licenses": [],
      "purl": "pkg:pypi/MarkupSafe@1.1.1"
    },
    {
      "type": "library",
      "name": "ItsDangerous",
      "version": "1.1.0",
      "description": "Safely pass trusted data to untrusted environments.",
      "licenses": [],
      "purl": "pkg:pypi/ItsDangerous@1.1.0"
    },
    {
      "type": "library",
      "name": "MySQL-python",
      "version": "1.2.5",
      "description": "Python interface to MySQL.",
      "licenses": [],
      "purl": "pkg:pypi/MySQL-python@1.2.5"
    }
  ]
}

payload_correct_bom_master3 = {
  "bomFormat": "CycloneDX",
  "specVersion": "1.4",
  "serialNumber": "urn:uuid:e8c355aa-2142-4084-a8c7-6d42c8610ba2",
  "version": 1,
  "metadata": {
    "timestamp": "2022-01-09T12:00:00Z",
    "component": {
      "type": "device",
      "bom-ref": "pcie-sata-adaptor-board",
      "group": "uk.ac.cam.cl",
      "name": "pcie-sata-adaptor-board",
      "version": "rev-1"
    }
  },
  "components": [
    {
      "type": "device",
      "bom-ref": "PCIE-098-02-F-D-EMS2",
      "supplier": {
        "name": "Samtec",
        "url": ["https://www.samtec.com/"]
      },
      "name": "PCIE-098-02-F-D-EMS2",
      "version": "2.9.10",
      "description": "Low Profile PCI Express® GEN 4 Connector",
      "externalReferences": [
        {
          "type": "documentation",
          "url": "https://www.samtec.com/products/pcie-098-02-f-d-ems2"
        }
      ],
      "properties": [
        {
          "name": "cdx:device:quantity",
          "value": "1"
        },
        {
          "name": "cdx:device:function",
          "value": "connector"
        },
        {
          "name": "cdx:device:location",
          "value": "mainboard"
        },
        {
          "name": "cdx:device:deviceType",
          "value": "thru-hole"
        }
      ]
    },
    {
      "type": "device",
      "bom-ref": "molex-47155-4001",
      "supplier": {
        "name": "Molex",
        "url": ["https://www.molex.com/"]
      },
      "name": "47155-4001",
      "description": "I/O Connectors HIGH SPEED CONN 1.27 VERT DIP SOLDERTAIL",
      "externalReferences": [
        {
          "type": "documentation",
          "url": "https://www.mouser.com/ProductDetail/Molex/47155-4001?qs=%2F%252BEk3ugtboRcLyK5WJikBg%3D%3D"
        }
      ],
      "properties": [
        {
          "name": "cdx:device:quantity",
          "value": "8"
        },
        {
          "name": "cdx:device:function",
          "value": "connector"
        },
        {
          "name": "cdx:device:location",
          "value": "mainboard"
        },
        {
          "name": "cdx:device:deviceType",
          "value": "thru-hole"
        },
        {
          "name": "cdx:device:gs1:gtin-12",
          "value": "822348522712"
        }
      ]
    },
    {
      "type": "device",
      "bom-ref": "3M-5622-4100-ML",
      "supplier": {
        "name": "3M",
        "url": ["https://www.3m.com/"]
      },
      "name": "5622-4100-ML",
      "description": "22 Position SATA Plug Connector Solder Through Hole, Right Angle",
      "externalReferences": [
        {
          "type": "documentation",
          "url": "https://www.3m.com/3M/en_US/p/d/b00036230/"
        }
      ],
      "properties": [
        {
          "name": "cdx:device:quantity",
          "value": "1"
        },
        {
          "name": "cdx:device:function",
          "value": "connector"
        },
        {
          "name": "cdx:device:location",
          "value": "mainboard"
        },
        {
          "name": "cdx:device:deviceType",
          "value": "thru-hole"
        }
      ]
    },
    {
      "type": "device",
      "bom-ref": "MC9A22-1034",
      "supplier": {
        "name": "Multicomp",
        "url": ["https://www.multicomp-pro.com/"]
      },
      "name": "MC9A22-1034",
      "description": "Header, Tht, Right Angle, 2.54MM, 10WAY",
      "externalReferences": [
        {
          "type": "documentation",
          "url": "https://octopart.com/datasheet/mc9a22-1034-multicomp-5403541"
        }
      ],
      "properties": [
        {
          "name": "cdx:device:quantity",
          "value": "1"
        },
        {
          "name": "cdx:device:function",
          "value": "connector"
        },
        {
          "name": "cdx:device:location",
          "value": "mainboard"
        },
        {
          "name": "cdx:device:deviceType",
          "value": "thru-hole"
        }
      ]
    },
    {
      "type": "device",
      "bom-ref": "826632-3",
      "supplier": {
        "name": "TE Connectivity",
        "url": ["https://www.te.com/"]
      },
      "name": "826632-3",
      "description": "AMPMODU Headers, PCB Mount Header, Vertical, Board-to-Board, 6 Position, .1 in [2.54 mm] Centerline, Breakaway",
      "externalReferences": [
        {
          "type": "documentation",
          "url": "https://www.te.com/usa-en/product-826632-3.html"
        }
      ],
      "properties": [
        {
          "name": "cdx:device:quantity",
          "value": "2"
        },
        {
          "name": "cdx:device:function",
          "value": "connector"
        },
        {
          "name": "cdx:device:location",
          "value": "mainboard"
        },
        {
          "name": "cdx:device:deviceType",
          "value": "thru-hole"
        }
      ]
    },
    {
      "type": "device",
      "bom-ref": "MP000003",
      "supplier": {
        "name": "Multicomp",
        "url": ["https://www.multicomp-pro.com/"]
      },
      "name": "MP000003",
      "description": "Resistor jumper, 0R, 0805",
      "properties": [
        {
          "name": "cdx:device:quantity",
          "value": "2"
        },
        {
          "name": "cdx:device:function",
          "value": "jumper"
        },
        {
          "name": "cdx:device:location",
          "value": "mainboard"
        },
        {
          "name": "cdx:device:deviceType",
          "value": "smd"
        }
      ]
    }
  ]
}

payload_correct_CCM3 = {
  "components": [
    {
      "bom-ref": "requirements-L1",
      "description": "requirements line 1: ExifRead==2.3.1",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/ExifRead/"
        }
      ],
      "name": "ExifRead",
      "purl": "pkg:pypi/exifread@2.3.1",
      "type": "library",
      "version": "2.3.1"
    },
    {
      "bom-ref": "requirements-L3",
      "description": "requirements line 3: Flask==1.1.2",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/Flask/"
        }
      ],
      "name": "Flask",
      "purl": "pkg:pypi/flask@1.1.2",
      "type": "library",
      "version": "1.1.2"
    },
    {
      "bom-ref": "requirements-L2",
      "description": "requirements line 2: Flask-SQLAlchemy==2.4.4",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/Flask-SQLAlchemy/"
        }
      ],
      "name": "Flask-SQLAlchemy",
      "purl": "pkg:pypi/flask-sqlalchemy@2.4.4",
      "type": "library",
      "version": "2.4.4"
    },
    {
      "bom-ref": "requirements-L4",
      "description": "requirements line 4: HTMLParser==0.0.2",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/HTMLParser/"
        }
      ],
      "name": "HTMLParser",
      "purl": "pkg:pypi/htmlparser@0.0.2",
      "type": "library",
      "version": "0.0.2"
    },
    {
      "bom-ref": "requirements-L5",
      "description": "requirements line 5: PIL==1.1.6",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/PIL/"
        }
      ],
      "name": "PIL",
      "purl": "pkg:pypi/pil@1.1.6",
      "type": "library",
      "version": "1.1.6"
    },
    {
      "bom-ref": "requirements-L6",
      "description": "requirements line 6: Pillow==9.0.1",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/Pillow/"
        }
      ],
      "name": "Pillow",
      "purl": "pkg:pypi/pillow@9.0.1",
      "type": "library",
      "version": "9.0.1"
    },
    {
      "bom-ref": "requirements-L7",
      "description": "requirements line 7: PyAudio==0.2.11",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/PyAudio/"
        }
      ],
      "name": "PyAudio",
      "purl": "pkg:pypi/pyaudio@0.2.11",
      "type": "library",
      "version": "0.2.11"
    },
    {
      "bom-ref": "requirements-L8",
      "description": "requirements line 8: PyAutoGUI==0.9.50",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/PyAutoGUI/"
        }
      ],
      "name": "PyAutoGUI",
      "purl": "pkg:pypi/pyautogui@0.9.50",
      "type": "library",
      "version": "0.9.50"
    },
    {
      "bom-ref": "requirements-L9",
      "description": "requirements line 9: PyPDF2==1.27.5",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/PyPDF2/"
        }
      ],
      "name": "PyPDF2",
      "purl": "pkg:pypi/pypdf2@1.27.5",
      "type": "library",
      "version": "1.27.5"
    },
    {
      "bom-ref": "requirements-L10",
      "description": "requirements line 10: SpeechRecognition==3.8.1",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/SpeechRecognition/"
        }
      ],
      "name": "SpeechRecognition",
      "purl": "pkg:pypi/speechrecognition@3.8.1",
      "type": "library",
      "version": "3.8.1"
    },
    {
      "bom-ref": "requirements-L11",
      "description": "requirements line 11: beautifulsoup4==4.9.1",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/beautifulsoup4/"
        }
      ],
      "name": "beautifulsoup4",
      "purl": "pkg:pypi/beautifulsoup4@4.9.1",
      "type": "library",
      "version": "4.9.1"
    },
    {
      "bom-ref": "requirements-L12",
      "description": "requirements line 12: certifi==2020.6.20",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/certifi/"
        }
      ],
      "name": "certifi",
      "purl": "pkg:pypi/certifi@2020.6.20",
      "type": "library",
      "version": "2020.6.20"
    },
    {
      "bom-ref": "requirements-L13",
      "description": "requirements line 13: chardet==3.0.4",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/chardet/"
        }
      ],
      "name": "chardet",
      "purl": "pkg:pypi/chardet@3.0.4",
      "type": "library",
      "version": "3.0.4"
    },
    {
      "bom-ref": "requirements-L14",
      "description": "requirements line 14: chromedriver-binary==85.0.4183.38.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/chromedriver-binary/"
        }
      ],
      "name": "chromedriver-binary",
      "purl": "pkg:pypi/chromedriver-binary@85.0.4183.38.0",
      "type": "library",
      "version": "85.0.4183.38.0"
    },
    {
      "bom-ref": "requirements-L15",
      "description": "requirements line 15: click==7.1.2",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/click/"
        }
      ],
      "name": "click",
      "purl": "pkg:pypi/click@7.1.2",
      "type": "library",
      "version": "7.1.2"
    },
    {
      "bom-ref": "requirements-L16",
      "description": "requirements line 16: cycler==0.10.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/cycler/"
        }
      ],
      "name": "cycler",
      "purl": "pkg:pypi/cycler@0.10.0",
      "type": "library",
      "version": "0.10.0"
    },
    {
      "bom-ref": "requirements-L17",
      "description": "requirements line 17: dnspython==2.0.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/dnspython/"
        }
      ],
      "name": "dnspython",
      "purl": "pkg:pypi/dnspython@2.0.0",
      "type": "library",
      "version": "2.0.0"
    },
    {
      "bom-ref": "requirements-L18",
      "description": "requirements line 18: emoji==0.6.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/emoji/"
        }
      ],
      "name": "emoji",
      "purl": "pkg:pypi/emoji@0.6.0",
      "type": "library",
      "version": "0.6.0"
    },
    {
      "bom-ref": "requirements-L19",
      "description": "requirements line 19: ffmpeg==1.4",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/ffmpeg/"
        }
      ],
      "name": "ffmpeg",
      "purl": "pkg:pypi/ffmpeg@1.4",
      "type": "library",
      "version": "1.4"
    },
    {
      "bom-ref": "requirements-L20",
      "description": "requirements line 20: ffpyplayer==4.3.1",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/ffpyplayer/"
        }
      ],
      "name": "ffpyplayer",
      "purl": "pkg:pypi/ffpyplayer@4.3.1",
      "type": "library",
      "version": "4.3.1"
    },
    {
      "bom-ref": "requirements-L21",
      "description": "requirements line 21: gTTS==2.1.1",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/gTTS/"
        }
      ],
      "name": "gTTS",
      "purl": "pkg:pypi/gtts@2.1.1",
      "type": "library",
      "version": "2.1.1"
    },
    {
      "bom-ref": "requirements-L22",
      "description": "requirements line 22: gensim==3.8.3",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/gensim/"
        }
      ],
      "name": "gensim",
      "purl": "pkg:pypi/gensim@3.8.3",
      "type": "library",
      "version": "3.8.3"
    },
    {
      "bom-ref": "requirements-L23",
      "description": "requirements line 23: geopy==2.0.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/geopy/"
        }
      ],
      "name": "geopy",
      "purl": "pkg:pypi/geopy@2.0.0",
      "type": "library",
      "version": "2.0.0"
    },
    {
      "bom-ref": "requirements-L24",
      "description": "requirements line 24: googletrans==3.0.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/googletrans/"
        }
      ],
      "name": "googletrans",
      "purl": "pkg:pypi/googletrans@3.0.0",
      "type": "library",
      "version": "3.0.0"
    },
    {
      "bom-ref": "requirements-L25",
      "description": "requirements line 25: idna==2.10",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/idna/"
        }
      ],
      "name": "idna",
      "purl": "pkg:pypi/idna@2.10",
      "type": "library",
      "version": "2.10"
    },
    {
      "bom-ref": "requirements-L26",
      "description": "requirements line 26: img2pdf==0.4.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/img2pdf/"
        }
      ],
      "name": "img2pdf",
      "purl": "pkg:pypi/img2pdf@0.4.0",
      "type": "library",
      "version": "0.4.0"
    },
    {
      "bom-ref": "requirements-L27",
      "description": "requirements line 27: jupyterlab==2.2.10",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/jupyterlab/"
        }
      ],
      "name": "jupyterlab",
      "purl": "pkg:pypi/jupyterlab@2.2.10",
      "type": "library",
      "version": "2.2.10"
    },
    {
      "bom-ref": "requirements-L28",
      "description": "requirements line 28: kiwisolver==1.2.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/kiwisolver/"
        }
      ],
      "name": "kiwisolver",
      "purl": "pkg:pypi/kiwisolver@1.2.0",
      "type": "library",
      "version": "1.2.0"
    },
    {
      "bom-ref": "requirements-L29",
      "description": "requirements line 29: lxml==4.6.5",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/lxml/"
        }
      ],
      "name": "lxml",
      "purl": "pkg:pypi/lxml@4.6.5",
      "type": "library",
      "version": "4.6.5"
    },
    {
      "bom-ref": "requirements-L30",
      "description": "requirements line 30: matplotlib==3.3.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/matplotlib/"
        }
      ],
      "name": "matplotlib",
      "purl": "pkg:pypi/matplotlib@3.3.0",
      "type": "library",
      "version": "3.3.0"
    },
    {
      "bom-ref": "requirements-L31",
      "description": "requirements line 31: model==0.6.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/model/"
        }
      ],
      "name": "model",
      "purl": "pkg:pypi/model@0.6.0",
      "type": "library",
      "version": "0.6.0"
    },
    {
      "bom-ref": "requirements-L32",
      "description": "requirements line 32: newspaper==0.1.0.7",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/newspaper/"
        }
      ],
      "name": "newspaper",
      "purl": "pkg:pypi/newspaper@0.1.0.7",
      "type": "library",
      "version": "0.1.0.7"
    },
    {
      "bom-ref": "requirements-L33",
      "description": "requirements line 33: nltk==3.5",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/nltk/"
        }
      ],
      "name": "nltk",
      "purl": "pkg:pypi/nltk@3.5",
      "type": "library",
      "version": "3.5"
    },
    {
      "bom-ref": "requirements-L34",
      "description": "requirements line 34: notebook==6.4.10",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/notebook/"
        }
      ],
      "name": "notebook",
      "purl": "pkg:pypi/notebook@6.4.10",
      "type": "library",
      "version": "6.4.10"
    },
    {
      "bom-ref": "requirements-L35",
      "description": "requirements line 35: numpy==1.19.1",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/numpy/"
        }
      ],
      "name": "numpy",
      "purl": "pkg:pypi/numpy@1.19.1",
      "type": "library",
      "version": "1.19.1"
    },
    {
      "bom-ref": "requirements-L36",
      "description": "requirements line 36: opencv-python==4.3.0.36",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/opencv-python/"
        }
      ],
      "name": "opencv-python",
      "purl": "pkg:pypi/opencv-python@4.3.0.36",
      "type": "library",
      "version": "4.3.0.36"
    },
    {
      "bom-ref": "requirements-L37",
      "description": "requirements line 37: pandas==1.0.5",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/pandas/"
        }
      ],
      "name": "pandas",
      "purl": "pkg:pypi/pandas@1.0.5",
      "type": "library",
      "version": "1.0.5"
    },
    {
      "bom-ref": "requirements-L38",
      "description": "requirements line 38: pathlib==1.0.1",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/pathlib/"
        }
      ],
      "name": "pathlib",
      "purl": "pkg:pypi/pathlib@1.0.1",
      "type": "library",
      "version": "1.0.1"
    },
    {
      "bom-ref": "requirements-L39",
      "description": "requirements line 39: proxytest==0.5.4",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/proxytest/"
        }
      ],
      "name": "proxytest",
      "purl": "pkg:pypi/proxytest@0.5.4",
      "type": "library",
      "version": "0.5.4"
    },
    {
      "bom-ref": "requirements-L40",
      "description": "requirements line 40: psutil==5.7.2",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/psutil/"
        }
      ],
      "name": "psutil",
      "purl": "pkg:pypi/psutil@5.7.2",
      "type": "library",
      "version": "5.7.2"
    },
    {
      "bom-ref": "requirements-L41",
      "description": "requirements line 41: py-notifier==0.1",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/py-notifier/"
        }
      ],
      "name": "py-notifier",
      "purl": "pkg:pypi/py-notifier@0.1",
      "type": "library",
      "version": "0.1"
    },
    {
      "bom-ref": "requirements-L42",
      "description": "requirements line 42: pycryptodome==3.9.8",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/pycryptodome/"
        }
      ],
      "name": "pycryptodome",
      "purl": "pkg:pypi/pycryptodome@3.9.8",
      "type": "library",
      "version": "3.9.8"
    },
    {
      "bom-ref": "requirements-L43",
      "description": "requirements line 43: pyparsing==2.4.7",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/pyparsing/"
        }
      ],
      "name": "pyparsing",
      "purl": "pkg:pypi/pyparsing@2.4.7",
      "type": "library",
      "version": "2.4.7"
    },
    {
      "bom-ref": "requirements-L44",
      "description": "requirements line 44: python-dateutil==2.8.1",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/python-dateutil/"
        }
      ],
      "name": "python-dateutil",
      "purl": "pkg:pypi/python-dateutil@2.8.1",
      "type": "library",
      "version": "2.8.1"
    },
    {
      "bom-ref": "requirements-L45",
      "description": "requirements line 45: pytz==2020.1",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/pytz/"
        }
      ],
      "name": "pytz",
      "purl": "pkg:pypi/pytz@2020.1",
      "type": "library",
      "version": "2020.1"
    },
    {
      "bom-ref": "requirements-L46",
      "description": "requirements line 46: requests==2.23.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/requests/"
        }
      ],
      "name": "requests",
      "purl": "pkg:pypi/requests@2.23.0",
      "type": "library",
      "version": "2.23.0"
    },
    {
      "bom-ref": "requirements-L47",
      "description": "requirements line 47: selenium==3.141.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/selenium/"
        }
      ],
      "name": "selenium",
      "purl": "pkg:pypi/selenium@3.141.0",
      "type": "library",
      "version": "3.141.0"
    },
    {
      "bom-ref": "requirements-L48",
      "description": "requirements line 48: six==1.15.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/six/"
        }
      ],
      "name": "six",
      "purl": "pkg:pypi/six@1.15.0",
      "type": "library",
      "version": "1.15.0"
    },
    {
      "bom-ref": "requirements-L49",
      "description": "requirements line 49: soupsieve==2.0.1",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/soupsieve/"
        }
      ],
      "name": "soupsieve",
      "purl": "pkg:pypi/soupsieve@2.0.1",
      "type": "library",
      "version": "2.0.1"
    },
    {
      "bom-ref": "requirements-L50",
      "description": "requirements line 50: sumeval==0.2.2",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/sumeval/"
        }
      ],
      "name": "sumeval",
      "purl": "pkg:pypi/sumeval@0.2.2",
      "type": "library",
      "version": "0.2.2"
    },
    {
      "bom-ref": "requirements-L51",
      "description": "requirements line 51: sumy==0.8.1",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/sumy/"
        }
      ],
      "name": "sumy",
      "purl": "pkg:pypi/sumy@0.8.1",
      "type": "library",
      "version": "0.8.1"
    },
    {
      "bom-ref": "requirements-L52",
      "description": "requirements line 52: tensorflow==2.6.4",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/tensorflow/"
        }
      ],
      "name": "tensorflow",
      "purl": "pkg:pypi/tensorflow@2.6.4",
      "type": "library",
      "version": "2.6.4"
    },
    {
      "bom-ref": "requirements-L53",
      "description": "requirements line 53: tqdm==4.48.2",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/tqdm/"
        }
      ],
      "name": "tqdm",
      "purl": "pkg:pypi/tqdm@4.48.2",
      "type": "library",
      "version": "4.48.2"
    },
    {
      "bom-ref": "requirements-L54",
      "description": "requirements line 54: tweepy==3.9.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/tweepy/"
        }
      ],
      "name": "tweepy",
      "purl": "pkg:pypi/tweepy@3.9.0",
      "type": "library",
      "version": "3.9.0"
    },
    {
      "bom-ref": "requirements-L55",
      "description": "requirements line 55: urllib3==1.26.5",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/urllib3/"
        }
      ],
      "name": "urllib3",
      "purl": "pkg:pypi/urllib3@1.26.5",
      "type": "library",
      "version": "1.26.5"
    },
    {
      "bom-ref": "requirements-L56",
      "description": "requirements line 56: utils==1.0.1",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/utils/"
        }
      ],
      "name": "utils",
      "purl": "pkg:pypi/utils@1.0.1",
      "type": "library",
      "version": "1.0.1"
    },
    {
      "bom-ref": "requirements-L57",
      "description": "requirements line 57: webdriver-manager==3.2.2",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/webdriver-manager/"
        }
      ],
      "name": "webdriver-manager",
      "purl": "pkg:pypi/webdriver-manager@3.2.2",
      "type": "library",
      "version": "3.2.2"
    },
    {
      "bom-ref": "requirements-L58",
      "description": "requirements line 58: wget==3.2",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/wget/"
        }
      ],
      "name": "wget",
      "purl": "pkg:pypi/wget@3.2",
      "type": "library",
      "version": "3.2"
    },
    {
      "bom-ref": "requirements-L59",
      "description": "requirements line 59: wikipedia==1.4.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/wikipedia/"
        }
      ],
      "name": "wikipedia",
      "purl": "pkg:pypi/wikipedia@1.4.0",
      "type": "library",
      "version": "1.4.0"
    },
    {
      "bom-ref": "requirements-L60",
      "description": "requirements line 60: win10toast==0.9",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/win10toast/"
        }
      ],
      "name": "win10toast",
      "purl": "pkg:pypi/win10toast@0.9",
      "type": "library",
      "version": "0.9"
    },
    {
      "bom-ref": "requirements-L61",
      "description": "requirements line 61: wordcloud==1.8.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/wordcloud/"
        }
      ],
      "name": "wordcloud",
      "purl": "pkg:pypi/wordcloud@1.8.0",
      "type": "library",
      "version": "1.8.0"
    },
    {
      "bom-ref": "requirements-L62",
      "description": "requirements line 62: xmltodict==0.12.0",
      "externalReferences": [
        {
          "comment": "implicit dist url",
          "type": "distribution",
          "url": "https://pypi.org/simple/xmltodict/"
        }
      ],
      "name": "xmltodict",
      "purl": "pkg:pypi/xmltodict@0.12.0",
      "type": "library",
      "version": "0.12.0"
    }
  ],
  "dependencies": [
    {
      "ref": "requirements-L1"
    },
    {
      "ref": "requirements-L10"
    },
    {
      "ref": "requirements-L11"
    },
    {
      "ref": "requirements-L12"
    },
    {
      "ref": "requirements-L13"
    },
    {
      "ref": "requirements-L14"
    },
    {
      "ref": "requirements-L15"
    },
    {
      "ref": "requirements-L16"
    },
    {
      "ref": "requirements-L17"
    },
    {
      "ref": "requirements-L18"
    },
    {
      "ref": "requirements-L19"
    },
    {
      "ref": "requirements-L2"
    },
    {
      "ref": "requirements-L20"
    },
    {
      "ref": "requirements-L21"
    },
    {
      "ref": "requirements-L22"
    },
    {
      "ref": "requirements-L23"
    },
    {
      "ref": "requirements-L24"
    },
    {
      "ref": "requirements-L25"
    },
    {
      "ref": "requirements-L26"
    },
    {
      "ref": "requirements-L27"
    },
    {
      "ref": "requirements-L28"
    },
    {
      "ref": "requirements-L29"
    },
    {
      "ref": "requirements-L3"
    },
    {
      "ref": "requirements-L30"
    },
    {
      "ref": "requirements-L31"
    },
    {
      "ref": "requirements-L32"
    },
    {
      "ref": "requirements-L33"
    },
    {
      "ref": "requirements-L34"
    },
    {
      "ref": "requirements-L35"
    },
    {
      "ref": "requirements-L36"
    },
    {
      "ref": "requirements-L37"
    },
    {
      "ref": "requirements-L38"
    },
    {
      "ref": "requirements-L39"
    },
    {
      "ref": "requirements-L4"
    },
    {
      "ref": "requirements-L40"
    },
    {
      "ref": "requirements-L41"
    },
    {
      "ref": "requirements-L42"
    },
    {
      "ref": "requirements-L43"
    },
    {
      "ref": "requirements-L44"
    },
    {
      "ref": "requirements-L45"
    },
    {
      "ref": "requirements-L46"
    },
    {
      "ref": "requirements-L47"
    },
    {
      "ref": "requirements-L48"
    },
    {
      "ref": "requirements-L49"
    },
    {
      "ref": "requirements-L5"
    },
    {
      "ref": "requirements-L50"
    },
    {
      "ref": "requirements-L51"
    },
    {
      "ref": "requirements-L52"
    },
    {
      "ref": "requirements-L53"
    },
    {
      "ref": "requirements-L54"
    },
    {
      "ref": "requirements-L55"
    },
    {
      "ref": "requirements-L56"
    },
    {
      "ref": "requirements-L57"
    },
    {
      "ref": "requirements-L58"
    },
    {
      "ref": "requirements-L59"
    },
    {
      "ref": "requirements-L6"
    },
    {
      "ref": "requirements-L60"
    },
    {
      "ref": "requirements-L61"
    },
    {
      "ref": "requirements-L62"
    },
    {
      "ref": "requirements-L7"
    },
    {
      "ref": "requirements-L8"
    },
    {
      "ref": "requirements-L9"
    }
  ],
  "metadata": {
    "timestamp": "2025-02-25T09:09:30.883540+00:00",
    "tools": [
      {
        "externalReferences": [
          {
            "type": "build-system",
            "url": "https://github.com/CycloneDX/cyclonedx-python/actions"
          },
          {
            "type": "distribution",
            "url": "https://pypi.org/project/cyclonedx-bom/"
          },
          {
            "type": "documentation",
            "url": "https://cyclonedx-bom-tool.readthedocs.io/"
          },
          {
            "type": "issue-tracker",
            "url": "https://github.com/CycloneDX/cyclonedx-python/issues"
          },
          {
            "type": "license",
            "url": "https://github.com/CycloneDX/cyclonedx-python/blob/main/LICENSE"
          },
          {
            "type": "release-notes",
            "url": "https://github.com/CycloneDX/cyclonedx-python/blob/main/CHANGELOG.md"
          },
          {
            "type": "vcs",
            "url": "https://github.com/CycloneDX/cyclonedx-python/"
          },
          {
            "type": "website",
            "url": "https://github.com/CycloneDX/cyclonedx-python/#readme"
          }
        ],
        "name": "cyclonedx-bom",
        "vendor": "CycloneDX",
        "version": "4.6.1"
      },
      {
        "externalReferences": [
          {
            "type": "build-system",
            "url": "https://github.com/CycloneDX/cyclonedx-python-lib/actions"
          },
          {
            "type": "distribution",
            "url": "https://pypi.org/project/cyclonedx-python-lib/"
          },
          {
            "type": "documentation",
            "url": "https://cyclonedx-python-library.readthedocs.io/"
          },
          {
            "type": "issue-tracker",
            "url": "https://github.com/CycloneDX/cyclonedx-python-lib/issues"
          },
          {
            "type": "license",
            "url": "https://github.com/CycloneDX/cyclonedx-python-lib/blob/main/LICENSE"
          },
          {
            "type": "release-notes",
            "url": "https://github.com/CycloneDX/cyclonedx-python-lib/blob/main/CHANGELOG.md"
          },
          {
            "type": "vcs",
            "url": "https://github.com/CycloneDX/cyclonedx-python-lib"
          },
          {
            "type": "website",
            "url": "https://github.com/CycloneDX/cyclonedx-python-lib/#readme"
          }
        ],
        "name": "cyclonedx-python-lib",
        "vendor": "CycloneDX",
        "version": "7.6.2"
      }
    ]
  },
  "serialNumber": "urn:uuid:0e210424-f7e9-48d6-a607-a79cbac24e75",
  "version": 1,
  "$schema": "http://cyclonedx.org/schema/bom-1.5.schema.json",
  "bomFormat": "CycloneDX",
  "specVersion": "1.5"
}

payload_correct_CCM4 = {
  "bomFormat" : "CycloneDX",
  "specVersion" : "1.5",
  "serialNumber" : "urn:uuid:dce5f002-f1f2-4809-bf9d-8813ae2efb79",
  "version" : 1,
  "metadata" : {
    "timestamp" : "2025-02-25T09:19:18Z",
    "tools" : [
      {
        "vendor" : "OWASP",
        "name" : "Dependency-Track",
        "version" : "4.12.1"
      }
    ],
    "component" : {
      "type" : "application",
      "bom-ref" : "82e592b4-56b8-455a-b3f6-70a966996277",
      "name" : "test",
      "version" : "1.0"
    }
  },
  "components" : [
    {
      "type" : "library",
      "bom-ref" : "47a41a51-64e6-4c16-8fd9-3934ebb145a2",
      "name" : "Pillow",
      "version" : "9.4.0",
      "description" : "requirements line 10: Pillow==9.4.0",
      "purl" : "pkg:pypi/pillow@9.4.0",
      "externalReferences" : [
        {
          "type" : "distribution",
          "url" : "https://pypi.org/simple/Pillow/",
          "comment" : "implicit dist url"
        }
      ]
    },
    {
      "type" : "library",
      "bom-ref" : "2f4d0675-18a3-4205-b3b0-3dfb814ca54f",
      "name" : "certifi",
      "version" : "2022.12.7",
      "description" : "requirements line 1: certifi==2022.12.7",
      "purl" : "pkg:pypi/certifi@2022.12.7",
      "externalReferences" : [
        {
          "type" : "distribution",
          "url" : "https://pypi.org/simple/certifi/",
          "comment" : "implicit dist url"
        }
      ]
    },
    {
      "type" : "library",
      "bom-ref" : "09cfcf87-a619-4eb6-9638-9ecd4fa76055",
      "name" : "charset-normalizer",
      "version" : "3.0.1",
      "description" : "requirements line 2: charset-normalizer==3.0.1",
      "purl" : "pkg:pypi/charset-normalizer@3.0.1",
      "externalReferences" : [
        {
          "type" : "distribution",
          "url" : "https://pypi.org/simple/charset-normalizer/",
          "comment" : "implicit dist url"
        }
      ]
    },
    {
      "type" : "library",
      "bom-ref" : "a9fef477-0d97-4b5a-8e4d-40529c78c838",
      "name" : "colorama",
      "version" : "0.4.6",
      "description" : "requirements line 3: colorama==0.4.6",
      "purl" : "pkg:pypi/colorama@0.4.6",
      "externalReferences" : [
        {
          "type" : "distribution",
          "url" : "https://pypi.org/simple/colorama/",
          "comment" : "implicit dist url"
        }
      ]
    },
    {
      "type" : "library",
      "bom-ref" : "9a642a54-1e89-4a01-a48e-3780b8d7ab1c",
      "name" : "decorator",
      "version" : "4.4.2",
      "description" : "requirements line 4: decorator==4.4.2",
      "purl" : "pkg:pypi/decorator@4.4.2",
      "externalReferences" : [
        {
          "type" : "distribution",
          "url" : "https://pypi.org/simple/decorator/",
          "comment" : "implicit dist url"
        }
      ]
    },
    {
      "type" : "library",
      "bom-ref" : "547b6f87-49a8-4552-88da-0d1ff7507d4c",
      "name" : "idna",
      "version" : "3.4",
      "description" : "requirements line 5: idna==3.4",
      "purl" : "pkg:pypi/idna@3.4",
      "externalReferences" : [
        {
          "type" : "distribution",
          "url" : "https://pypi.org/simple/idna/",
          "comment" : "implicit dist url"
        }
      ]
    },
    {
      "type" : "library",
      "bom-ref" : "c06911a1-05be-41f5-af67-f9405b726bea",
      "name" : "imageio",
      "version" : "2.25.0",
      "description" : "requirements line 6: imageio==2.25.0",
      "purl" : "pkg:pypi/imageio@2.25.0",
      "externalReferences" : [
        {
          "type" : "distribution",
          "url" : "https://pypi.org/simple/imageio/",
          "comment" : "implicit dist url"
        }
      ]
    },
    {
      "type" : "library",
      "bom-ref" : "d6069bcc-0443-4f7d-a4ac-eceaeff92f5e",
      "name" : "imageio-ffmpeg",
      "version" : "0.4.8",
      "description" : "requirements line 7: imageio-ffmpeg==0.4.8",
      "purl" : "pkg:pypi/imageio-ffmpeg@0.4.8",
      "externalReferences" : [
        {
          "type" : "distribution",
          "url" : "https://pypi.org/simple/imageio-ffmpeg/",
          "comment" : "implicit dist url"
        }
      ]
    },
    {
      "type" : "library",
      "bom-ref" : "b0013349-0ec0-4d7e-868d-7cf860d0f05e",
      "name" : "moviepy",
      "version" : "1.0.3",
      "description" : "requirements line 8: moviepy==1.0.3",
      "purl" : "pkg:pypi/moviepy@1.0.3",
      "externalReferences" : [
        {
          "type" : "distribution",
          "url" : "https://pypi.org/simple/moviepy/",
          "comment" : "implicit dist url"
        }
      ]
    },
    {
      "type" : "library",
      "bom-ref" : "18282ac2-d895-4387-8ba7-2d068f0d1bbd",
      "name" : "numpy",
      "version" : "1.24.1",
      "description" : "requirements line 9: numpy==1.24.1",
      "purl" : "pkg:pypi/numpy@1.24.1",
      "externalReferences" : [
        {
          "type" : "distribution",
          "url" : "https://pypi.org/simple/numpy/",
          "comment" : "implicit dist url"
        }
      ]
    },
    {
      "type" : "library",
      "bom-ref" : "c254814a-a64b-4726-9c9f-50fb734023f9",
      "name" : "proglog",
      "version" : "0.1.10",
      "description" : "requirements line 11: proglog==0.1.10",
      "purl" : "pkg:pypi/proglog@0.1.10",
      "externalReferences" : [
        {
          "type" : "distribution",
          "url" : "https://pypi.org/simple/proglog/",
          "comment" : "implicit dist url"
        }
      ]
    },
    {
      "type" : "library",
      "bom-ref" : "f45d1cd6-2c59-42e6-899c-fbdb69b74d64",
      "name" : "requests",
      "version" : "2.28.2",
      "description" : "requirements line 12: requests==2.28.2",
      "purl" : "pkg:pypi/requests@2.28.2",
      "externalReferences" : [
        {
          "type" : "distribution",
          "url" : "https://pypi.org/simple/requests/",
          "comment" : "implicit dist url"
        }
      ]
    },
    {
      "type" : "library",
      "bom-ref" : "f37629f1-b3af-4a21-98c5-0059cf9cfdf3",
      "name" : "tk",
      "version" : "0.1.0",
      "description" : "requirements line 13: tk==0.1.0",
      "purl" : "pkg:pypi/tk@0.1.0",
      "externalReferences" : [
        {
          "type" : "distribution",
          "url" : "https://pypi.org/simple/tk/",
          "comment" : "implicit dist url"
        }
      ]
    },
    {
      "type" : "library",
      "bom-ref" : "f03b8152-604a-4b50-af8b-5d34b47f263c",
      "name" : "tqdm",
      "version" : "4.64.1",
      "description" : "requirements line 14: tqdm==4.64.1",
      "purl" : "pkg:pypi/tqdm@4.64.1",
      "externalReferences" : [
        {
          "type" : "distribution",
          "url" : "https://pypi.org/simple/tqdm/",
          "comment" : "implicit dist url"
        }
      ]
    },
    {
      "type" : "library",
      "bom-ref" : "2bf92814-a7e0-47b7-ae71-d8ceaa5880ae",
      "name" : "urllib3",
      "version" : "1.26.14",
      "description" : "requirements line 15: urllib3==1.26.14",
      "purl" : "pkg:pypi/urllib3@1.26.14",
      "externalReferences" : [
        {
          "type" : "distribution",
          "url" : "https://pypi.org/simple/urllib3/",
          "comment" : "implicit dist url"
        }
      ]
    }
  ],
  "dependencies" : [
    {
      "ref" : "82e592b4-56b8-455a-b3f6-70a966996277",
      "dependsOn" : [ ]
    },
    {
      "ref" : "47a41a51-64e6-4c16-8fd9-3934ebb145a2",
      "dependsOn" : [ ]
    },
    {
      "ref" : "2f4d0675-18a3-4205-b3b0-3dfb814ca54f",
      "dependsOn" : [ ]
    },
    {
      "ref" : "09cfcf87-a619-4eb6-9638-9ecd4fa76055",
      "dependsOn" : [ ]
    },
    {
      "ref" : "a9fef477-0d97-4b5a-8e4d-40529c78c838",
      "dependsOn" : [ ]
    },
    {
      "ref" : "9a642a54-1e89-4a01-a48e-3780b8d7ab1c",
      "dependsOn" : [ ]
    },
    {
      "ref" : "547b6f87-49a8-4552-88da-0d1ff7507d4c",
      "dependsOn" : [ ]
    },
    {
      "ref" : "c06911a1-05be-41f5-af67-f9405b726bea",
      "dependsOn" : [ ]
    },
    {
      "ref" : "d6069bcc-0443-4f7d-a4ac-eceaeff92f5e",
      "dependsOn" : [ ]
    },
    {
      "ref" : "b0013349-0ec0-4d7e-868d-7cf860d0f05e",
      "dependsOn" : [ ]
    },
    {
      "ref" : "18282ac2-d895-4387-8ba7-2d068f0d1bbd",
      "dependsOn" : [ ]
    },
    {
      "ref" : "c254814a-a64b-4726-9c9f-50fb734023f9",
      "dependsOn" : [ ]
    },
    {
      "ref" : "f45d1cd6-2c59-42e6-899c-fbdb69b74d64",
      "dependsOn" : [ ]
    },
    {
      "ref" : "f37629f1-b3af-4a21-98c5-0059cf9cfdf3",
      "dependsOn" : [ ]
    },
    {
      "ref" : "f03b8152-604a-4b50-af8b-5d34b47f263c",
      "dependsOn" : [ ]
    },
    {
      "ref" : "2bf92814-a7e0-47b7-ae71-d8ceaa5880ae",
      "dependsOn" : [ ]
    }
  ],
  "vulnerabilities" : [
    {
      "bom-ref" : "bcd6ed5f-338c-4a2e-8109-1c999bf77909",
      "id" : "CVE-2023-44271",
      "source" : {
        "name" : "NVD",
        "url" : "https://nvd.nist.gov/"
      },
      "ratings" : [
        {
          "source" : {
            "name" : "NVD",
            "url" : "https://nvd.nist.gov/"
          },
          "score" : 7.5,
          "severity" : "high",
          "method" : "CVSSv3",
          "vector" : "CVSS:3.0/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:N/A:H"
        }
      ],
      "cwes" : [
        770
      ],
      "description" : "An issue was discovered in Pillow before 10.0.0. It is a Denial of Service that uncontrollably allocates memory to process a given task, potentially causing a service to crash by having it run out of memory. This occurs for truetype in ImageFont when textlength in an ImageDraw instance operates on a long text argument.",
      "published" : "2023-11-03T05:15:00Z",
      "updated" : "2024-11-21T08:25:00Z",
      "affects" : [
        {
          "ref" : "47a41a51-64e6-4c16-8fd9-3934ebb145a2"
        }
      ]
    },
    {
      "bom-ref" : "887673cd-4dc9-426e-ac73-e3c3bb20cab9",
      "id" : "CVE-2023-50447",
      "source" : {
        "name" : "NVD",
        "url" : "https://nvd.nist.gov/"
      },
      "ratings" : [
        {
          "source" : {
            "name" : "NVD",
            "url" : "https://nvd.nist.gov/"
          },
          "score" : 8.1,
          "severity" : "high",
          "method" : "CVSSv3",
          "vector" : "CVSS:3.0/AV:N/AC:H/PR:N/UI:N/S:U/C:H/I:H/A:H"
        }
      ],
      "cwes" : [
        94
      ],
      "description" : "Pillow through 10.1.0 allows PIL.ImageMath.eval Arbitrary Code Execution via the environment parameter, a different vulnerability than CVE-2022-22817 (which was about the expression parameter).",
      "published" : "2024-01-19T20:15:00Z",
      "updated" : "2024-11-21T08:37:00Z",
      "affects" : [
        {
          "ref" : "47a41a51-64e6-4c16-8fd9-3934ebb145a2"
        }
      ]
    },
    {
      "bom-ref" : "9940c65a-2d42-433b-b2fc-8b23d3e44b6f",
      "id" : "CVE-2024-28219",
      "source" : {
        "name" : "NVD",
        "url" : "https://nvd.nist.gov/"
      },
      "ratings" : [
        {
          "source" : {
            "name" : "NVD",
            "url" : "https://nvd.nist.gov/"
          },
          "severity" : "unknown",
          "method" : "other"
        }
      ],
      "description" : "In _imagingcms.c in Pillow before 10.3.0, a buffer overflow exists because strcpy is used instead of strncpy.",
      "published" : "2024-04-03T03:15:00Z",
      "updated" : "2024-11-21T09:06:00Z",
      "affects" : [
        {
          "ref" : "47a41a51-64e6-4c16-8fd9-3934ebb145a2"
        }
      ]
    },
    {
      "bom-ref" : "dd48aea9-abf8-4066-a302-f9fdafde5f94",
      "id" : "CVE-2023-37920",
      "source" : {
        "name" : "NVD",
        "url" : "https://nvd.nist.gov/"
      },
      "ratings" : [
        {
          "source" : {
            "name" : "NVD",
            "url" : "https://nvd.nist.gov/"
          },
          "score" : 9.8,
          "severity" : "critical",
          "method" : "CVSSv3",
          "vector" : "CVSS:3.0/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H"
        }
      ],
      "cwes" : [
        345
      ],
      "description" : "Certifi is a curated collection of Root Certificates for validating the trustworthiness of SSL certificates while verifying the identity of TLS hosts. Certifi prior to version 2023.07.22 recognizes \"e-Tugra\" root certificates. e-Tugra's root certificates were subject to an investigation prompted by reporting of security issues in their systems. Certifi 2023.07.22 removes root certificates from \"e-Tugra\" from the root store.",
      "published" : "2023-07-25T21:15:00Z",
      "updated" : "2025-02-13T13:50:00Z",
      "affects" : [
        {
          "ref" : "2f4d0675-18a3-4205-b3b0-3dfb814ca54f"
        }
      ]
    },
    {
      "bom-ref" : "5da6c82a-aef0-4553-bdae-f44e8baec6e7",
      "id" : "CVE-2024-39689",
      "source" : {
        "name" : "NVD",
        "url" : "https://nvd.nist.gov/"
      },
      "ratings" : [
        {
          "source" : {
            "name" : "NVD",
            "url" : "https://nvd.nist.gov/"
          },
          "score" : 7.5,
          "severity" : "high",
          "method" : "CVSSv3",
          "vector" : "CVSS:3.0/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:H/A:N"
        }
      ],
      "cwes" : [
        345
      ],
      "description" : "Certifi is a curated collection of Root Certificates for validating the trustworthiness of SSL certificates while verifying the identity of TLS hosts. Certifi starting in 2021.5.30 and prior to 2024.7.4 recognized root certificates from `GLOBALTRUST`. Certifi 2024.7.04 removes root certificates from `GLOBALTRUST` from the root store. These are in the process of being removed from Mozilla's trust store. `GLOBALTRUST`'s root certificates are being removed pursuant to an investigation which identified \"long-running and unresolved compliance issues.\"",
      "published" : "2024-07-05T19:15:00Z",
      "updated" : "2025-02-15T00:15:00Z",
      "affects" : [
        {
          "ref" : "2f4d0675-18a3-4205-b3b0-3dfb814ca54f"
        }
      ]
    },
    {
      "bom-ref" : "cba0c835-5170-43dc-9079-f971aa3098dd",
      "id" : "CVE-2024-3651",
      "source" : {
        "name" : "NVD",
        "url" : "https://nvd.nist.gov/"
      },
      "ratings" : [
        {
          "source" : {
            "name" : "NVD",
            "url" : "https://nvd.nist.gov/"
          },
          "score" : 7.5,
          "severity" : "high",
          "method" : "CVSSv3",
          "vector" : "CVSS:3.0/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:N/A:H"
        }
      ],
      "description" : "A vulnerability was identified in the kjd/idna library, specifically within the `idna.encode()` function, affecting version 3.6. The issue arises from the function's handling of crafted input strings, which can lead to quadratic complexity and consequently, a denial of service condition. This vulnerability is triggered by a crafted input that causes the `idna.encode()` function to process the input with considerable computational load, significantly increasing the processing time in a quadratic manner relative to the input size.",
      "published" : "2024-07-07T18:15:00Z",
      "updated" : "2024-11-21T09:30:00Z",
      "affects" : [
        {
          "ref" : "547b6f87-49a8-4552-88da-0d1ff7507d4c"
        }
      ]
    },
    {
      "bom-ref" : "22105a0a-361d-486b-af48-49d80fcde3c7",
      "id" : "CVE-2023-32681",
      "source" : {
        "name" : "NVD",
        "url" : "https://nvd.nist.gov/"
      },
      "ratings" : [
        {
          "source" : {
            "name" : "NVD",
            "url" : "https://nvd.nist.gov/"
          },
          "score" : 6.1,
          "severity" : "medium",
          "method" : "CVSSv3",
          "vector" : "CVSS:3.0/AV:N/AC:H/PR:N/UI:R/S:C/C:H/I:N/A:N"
        }
      ],
      "description" : "Requests is a HTTP library. Since Requests 2.3.0, Requests has been leaking Proxy-Authorization headers to destination servers when redirected to an HTTPS endpoint. This is a product of how we use `rebuild_proxies` to reattach the `Proxy-Authorization` header to requests. For HTTP connections sent through the tunnel, the proxy will identify the header in the request itself and remove it prior to forwarding to the destination server. However when sent over HTTPS, the `Proxy-Authorization` header must be sent in the CONNECT request as the proxy has no visibility into the tunneled request. This results in Requests forwarding proxy credentials to the destination server unintentionally, allowing a malicious actor to potentially exfiltrate sensitive information. This issue has been patched in version 2.31.0.",
      "published" : "2023-05-26T18:15:00Z",
      "updated" : "2025-02-13T17:16:00Z",
      "affects" : [
        {
          "ref" : "f45d1cd6-2c59-42e6-899c-fbdb69b74d64"
        }
      ]
    },
    {
      "bom-ref" : "fc0210ab-f1a3-44db-8cb4-69825bbcf39b",
      "id" : "CVE-2024-35195",
      "source" : {
        "name" : "NVD",
        "url" : "https://nvd.nist.gov/"
      },
      "ratings" : [
        {
          "source" : {
            "name" : "NVD",
            "url" : "https://nvd.nist.gov/"
          },
          "severity" : "unknown",
          "method" : "other"
        }
      ],
      "description" : "Requests is a HTTP library. Prior to 2.32.0, when making requests through a Requests `Session`, if the first request is made with `verify=False` to disable cert verification, all subsequent requests to the same host will continue to ignore cert verification regardless of changes to the value of `verify`. This behavior will continue for the lifecycle of the connection in the connection pool. This vulnerability is fixed in 2.32.0.",
      "published" : "2024-05-20T21:15:00Z",
      "updated" : "2024-11-21T09:19:00Z",
      "affects" : [
        {
          "ref" : "f45d1cd6-2c59-42e6-899c-fbdb69b74d64"
        }
      ]
    },
    {
      "bom-ref" : "c0c11cda-82af-42ee-aff8-c59df8e069af",
      "id" : "CVE-2024-34062",
      "source" : {
        "name" : "NVD",
        "url" : "https://nvd.nist.gov/"
      },
      "ratings" : [
        {
          "source" : {
            "name" : "NVD",
            "url" : "https://nvd.nist.gov/"
          },
          "severity" : "unknown",
          "method" : "other"
        }
      ],
      "description" : "tqdm is an open source progress bar for Python and CLI. Any optional non-boolean CLI arguments (e.g. `--delim`, `--buf-size`, `--manpath`) are passed through python's `eval`, allowing arbitrary code execution. This issue is only locally exploitable and had been addressed in release version 4.66.3. All users are advised to upgrade. There are no known workarounds for this vulnerability.",
      "published" : "2024-05-03T10:15:00Z",
      "updated" : "2024-11-21T09:18:00Z",
      "affects" : [
        {
          "ref" : "f03b8152-604a-4b50-af8b-5d34b47f263c"
        }
      ]
    },
    {
      "bom-ref" : "f4cf781b-de16-439c-a8d8-927d7c87aef4",
      "id" : "CVE-2023-43804",
      "source" : {
        "name" : "NVD",
        "url" : "https://nvd.nist.gov/"
      },
      "ratings" : [
        {
          "source" : {
            "name" : "NVD",
            "url" : "https://nvd.nist.gov/"
          },
          "score" : 8.1,
          "severity" : "high",
          "method" : "CVSSv3",
          "vector" : "CVSS:3.0/AV:N/AC:L/PR:L/UI:N/S:U/C:H/I:H/A:N"
        }
      ],
      "description" : "urllib3 is a user-friendly HTTP client library for Python. urllib3 doesn't treat the `Cookie` HTTP header special or provide any helpers for managing cookies over HTTP, that is the responsibility of the user. However, it is possible for a user to specify a `Cookie` header and unknowingly leak information via HTTP redirects to a different origin if that user doesn't disable redirects explicitly. This issue has been patched in urllib3 version 1.26.17 or 2.0.5.",
      "published" : "2023-10-04T17:15:00Z",
      "updated" : "2024-12-13T14:15:00Z",
      "affects" : [
        {
          "ref" : "2bf92814-a7e0-47b7-ae71-d8ceaa5880ae"
        }
      ]
    },
    {
      "bom-ref" : "a05f5f5c-8a51-4604-92e8-40884ec82ca6",
      "id" : "CVE-2023-45803",
      "source" : {
        "name" : "NVD",
        "url" : "https://nvd.nist.gov/"
      },
      "ratings" : [
        {
          "source" : {
            "name" : "NVD",
            "url" : "https://nvd.nist.gov/"
          },
          "score" : 4.2,
          "severity" : "medium",
          "method" : "CVSSv3",
          "vector" : "CVSS:3.0/AV:A/AC:H/PR:H/UI:N/S:U/C:H/I:N/A:N"
        }
      ],
      "description" : "urllib3 is a user-friendly HTTP client library for Python. urllib3 previously wouldn't remove the HTTP request body when an HTTP redirect response using status 301, 302, or 303 after the request had its method changed from one that could accept a request body (like `POST`) to `GET` as is required by HTTP RFCs. Although this behavior is not specified in the section for redirects, it can be inferred by piecing together information from different sections and we have observed the behavior in other major HTTP client implementations like curl and web browsers. Because the vulnerability requires a previously trusted service to become compromised in order to have an impact on confidentiality we believe the exploitability of this vulnerability is low. Additionally, many users aren't putting sensitive data in HTTP request bodies, if this is the case then this vulnerability isn't exploitable. Both of the following conditions must be true to be affected by this vulnerability: 1. Using urllib3 and submitting sensitive information in the HTTP request body (such as form data or JSON) and 2. The origin service is compromised and starts redirecting using 301, 302, or 303 to a malicious peer or the redirected-to service becomes compromised. This issue has been addressed in versions 1.26.18 and 2.0.7 and users are advised to update to resolve this issue. Users unable to update should disable redirects for services that aren't expecting to respond with redirects with `redirects=False` and disable automatic redirects with `redirects=False` and handle 301, 302, and 303 redirects manually by stripping the HTTP request body.",
      "published" : "2023-10-17T20:15:00Z",
      "updated" : "2025-02-13T18:15:00Z",
      "affects" : [
        {
          "ref" : "2bf92814-a7e0-47b7-ae71-d8ceaa5880ae"
        }
      ]
    }
  ]
}

payload_correct_CCM5 = {
  "oid_refs": {
    "aes-256-ccm": {
      "algo_oid": "2.16.840.1.101.3.4.1.47"
    },
    "id-aes256-CCM": {
      "algo_oid": "N/A"
    },
    "AES-128-OFB": {
      "algo_oid": "2.16.840.1.101.3.4.1.3"
    },
    "AES-128-ECB": {
      "algo_oid": "2.16.840.1.101.3.4.1.1"
    },
    "ARIA-256-CBC": {
      "algo_oid": "1.2.410.200046.1.1.12"
    },
    "ARIA256": {
      "algo_oid": "N/A"
    },
    "aes-256-gcm": {
      "algo_oid": "2.16.840.1.101.3.4.1.46"
    },
    "id-aes256-GCM": {
      "algo_oid": "N/A"
    },
    "ARIA-128-GCM": {
      "algo_oid": "1.2.410.200046.1.1.34"
    },
    "SM4-ECB": {
      "algo_oid": "1.2.156.10197.1.104.1"
    },
    "AES-192-CFB": {
      "algo_oid": "2.16.840.1.101.3.4.1.24"
    },
    "ARIA-256-ECB": {
      "algo_oid": "1.2.410.200046.1.1.11"
    },
    "CAMELLIA-128-CBC": {
      "algo_oid": "1.2.392.200011.61.1.1.1.2"
    },
    "CAMELLIA128": {
      "algo_oid": "N/A"
    },
    "AES-128-WRAP": {
      "algo_oid": "2.16.840.1.101.3.4.1.5"
    },
    "AES128-WRAP": {
      "algo_oid": "N/A"
    },
    "id-aes128-wrap": {
      "algo_oid": "N/A"
    },
    "DES3-WRAP": {
      "algo_oid": "1.2.840.113549.1.9.16.3.6"
    },
    "id-smime-alg-CMS3DESwrap": {
      "algo_oid": "N/A"
    },
    "CAMELLIA-128-CTR": {
      "algo_oid": "0.3.4401.5.3.1.9.9"
    },
    "SM4-OFB": {
      "algo_oid": "1.2.156.10197.1.104.3"
    },
    "SM4-OFB128": {
      "algo_oid": "N/A"
    },
    "CAMELLIA-192-CFB": {
      "algo_oid": "0.3.4401.5.3.1.9.24"
    },
    "ARIA-256-CCM": {
      "algo_oid": "1.2.410.200046.1.1.39"
    },
    "AES-128-XTS": {
      "algo_oid": "1.3.111.2.1619.0.1.1"
    },
    "ARIA-256-GCM": {
      "algo_oid": "1.2.410.200046.1.1.36"
    },
    "AES-128-CFB": {
      "algo_oid": "2.16.840.1.101.3.4.1.4"
    },
    "aes-128-gcm": {
      "algo_oid": "2.16.840.1.101.3.4.1.6"
    },
    "id-aes128-GCM": {
      "algo_oid": "N/A"
    },
    "AES-256-CBC": {
      "algo_oid": "2.16.840.1.101.3.4.1.42"
    },
    "AES256": {
      "algo_oid": "N/A"
    },
    "DES-EDE3-CBC": {
      "algo_oid": "1.2.840.113549.3.7"
    },
    "DES3": {
      "algo_oid": "N/A"
    },
    "CAMELLIA-256-CFB": {
      "algo_oid": "0.3.4401.5.3.1.9.44"
    },
    "CAMELLIA-128-CFB": {
      "algo_oid": "0.3.4401.5.3.1.9.4"
    },
    "CAMELLIA-192-ECB": {
      "algo_oid": "0.3.4401.5.3.1.9.21"
    },
    "AES-256-WRAP": {
      "algo_oid": "2.16.840.1.101.3.4.1.45"
    },
    "AES256-WRAP": {
      "algo_oid": "N/A"
    },
    "id-aes256-wrap": {
      "algo_oid": "N/A"
    },
    "aes-192-ccm": {
      "algo_oid": "2.16.840.1.101.3.4.1.27"
    },
    "id-aes192-CCM": {
      "algo_oid": "N/A"
    },
    "ARIA-192-ECB": {
      "algo_oid": "1.2.410.200046.1.1.6"
    },
    "AES-256-OFB": {
      "algo_oid": "2.16.840.1.101.3.4.1.43"
    },
    "CAMELLIA-192-CTR": {
      "algo_oid": "0.3.4401.5.3.1.9.29"
    },
    "AES-256-CFB": {
      "algo_oid": "2.16.840.1.101.3.4.1.44"
    },
    "aes-128-ccm": {
      "algo_oid": "2.16.840.1.101.3.4.1.7"
    },
    "id-aes128-CCM": {
      "algo_oid": "N/A"
    },
    "aes-192-gcm": {
      "algo_oid": "2.16.840.1.101.3.4.1.26"
    },
    "id-aes192-GCM": {
      "algo_oid": "N/A"
    },
    "AES-128-WRAP-PAD": {
      "algo_oid": "2.16.840.1.101.3.4.1.8"
    },
    "AES128-WRAP-PAD": {
      "algo_oid": "N/A"
    },
    "id-aes128-wrap-pad": {
      "algo_oid": "N/A"
    },
    "ARIA-128-CTR": {
      "algo_oid": "1.2.410.200046.1.1.5"
    },
    "AES-256-ECB": {
      "algo_oid": "2.16.840.1.101.3.4.1.41"
    },
    "ARIA-128-OFB": {
      "algo_oid": "1.2.410.200046.1.1.4"
    },
    "ARIA-256-CFB": {
      "algo_oid": "1.2.410.200046.1.1.13"
    },
    "ARIA-192-GCM": {
      "algo_oid": "1.2.410.200046.1.1.35"
    },
    "CAMELLIA-128-ECB": {
      "algo_oid": "0.3.4401.5.3.1.9.1"
    },
    "CAMELLIA-192-CBC": {
      "algo_oid": "1.2.392.200011.61.1.1.1.3"
    },
    "CAMELLIA192": {
      "algo_oid": "N/A"
    },
    "AES-192-CBC": {
      "algo_oid": "2.16.840.1.101.3.4.1.22"
    },
    "AES192": {
      "algo_oid": "N/A"
    },
    "ARIA-192-CBC": {
      "algo_oid": "1.2.410.200046.1.1.7"
    },
    "ARIA192": {
      "algo_oid": "N/A"
    },
    "AES-192-ECB": {
      "algo_oid": "2.16.840.1.101.3.4.1.21"
    },
    "ARIA-128-CFB": {
      "algo_oid": "1.2.410.200046.1.1.3"
    },
    "ARIA-256-CTR": {
      "algo_oid": "1.2.410.200046.1.1.15"
    },
    "SM4": {
      "algo_oid": "1.2.156.10197.1.104.2"
    },
    "SM4-CBC": {
      "algo_oid": "N/A"
    },
    "AES-256-WRAP-PAD": {
      "algo_oid": "2.16.840.1.101.3.4.1.48"
    },
    "AES256-WRAP-PAD": {
      "algo_oid": "N/A"
    },
    "id-aes256-wrap-pad": {
      "algo_oid": "N/A"
    },
    "AES-192-WRAP": {
      "algo_oid": "2.16.840.1.101.3.4.1.25"
    },
    "AES192-WRAP": {
      "algo_oid": "N/A"
    },
    "id-aes192-wrap": {
      "algo_oid": "N/A"
    },
    "SM4-CTR": {
      "algo_oid": "1.2.156.10197.1.104.7"
    },
    "CAMELLIA-192-OFB": {
      "algo_oid": "0.3.4401.5.3.1.9.23"
    },
    "AES-128-CBC": {
      "algo_oid": "2.16.840.1.101.3.4.1.2"
    },
    "AES128": {
      "algo_oid": "N/A"
    },
    "CAMELLIA-256-OFB": {
      "algo_oid": "0.3.4401.5.3.1.9.43"
    },
    "CAMELLIA-256-CBC": {
      "algo_oid": "1.2.392.200011.61.1.1.1.4"
    },
    "CAMELLIA256": {
      "algo_oid": "N/A"
    },
    "SM4-CFB": {
      "algo_oid": "1.2.156.10197.1.104.4"
    },
    "SM4-CFB128": {
      "algo_oid": "N/A"
    },
    "ARIA-128-CBC": {
      "algo_oid": "1.2.410.200046.1.1.2"
    },
    "ARIA128": {
      "algo_oid": "N/A"
    },
    "CAMELLIA-128-OFB": {
      "algo_oid": "0.3.4401.5.3.1.9.3"
    },
    "ARIA-192-CCM": {
      "algo_oid": "1.2.410.200046.1.1.38"
    },
    "ARIA-256-OFB": {
      "algo_oid": "1.2.410.200046.1.1.14"
    },
    "CAMELLIA-256-CTR": {
      "algo_oid": "0.3.4401.5.3.1.9.49"
    },
    "DES-EDE": {
      "algo_oid": "1.3.14.3.2.17"
    },
    "DES-EDE-ECB": {
      "algo_oid": "N/A"
    },
    "AES-192-OFB": {
      "algo_oid": "2.16.840.1.101.3.4.1.23"
    },
    "ARIA-192-CTR": {
      "algo_oid": "1.2.410.200046.1.1.10"
    },
    "CAMELLIA-256-ECB": {
      "algo_oid": "0.3.4401.5.3.1.9.41"
    },
    "ARIA-128-ECB": {
      "algo_oid": "1.2.410.200046.1.1.1"
    },
    "ARIA-192-CFB": {
      "algo_oid": "1.2.410.200046.1.1.8"
    },
    "ARIA-192-OFB": {
      "algo_oid": "1.2.410.200046.1.1.9"
    },
    "ARIA-128-CCM": {
      "algo_oid": "1.2.410.200046.1.1.37"
    },
    "AES-256-XTS": {
      "algo_oid": "1.3.111.2.1619.0.1.2"
    },
    "AES-192-WRAP-PAD": {
      "algo_oid": "2.16.840.1.101.3.4.1.28"
    },
    "AES192-WRAP-PAD": {
      "algo_oid": "N/A"
    },
    "id-aes192-wrap-pad": {
      "algo_oid": "N/A"
    },
    "AES-256-WRAP-INV": {
      "algo_oid": "N/A"
    },
    "AES256-WRAP-INV": {
      "algo_oid": "N/A"
    },
    "AES-192-WRAP-INV": {
      "algo_oid": "N/A"
    },
    "AES192-WRAP-INV": {
      "algo_oid": "N/A"
    },
    "AES-128-WRAP-INV": {
      "algo_oid": "N/A"
    },
    "AES128-WRAP-INV": {
      "algo_oid": "N/A"
    },
    "AES-256-WRAP-PAD-INV": {
      "algo_oid": "N/A"
    },
    "AES256-WRAP-PAD-INV": {
      "algo_oid": "N/A"
    },
    "AES-192-WRAP-PAD-INV": {
      "algo_oid": "N/A"
    },
    "AES192-WRAP-PAD-INV": {
      "algo_oid": "N/A"
    },
    "AES-128-WRAP-PAD-INV": {
      "algo_oid": "N/A"
    },
    "AES128-WRAP-PAD-INV": {
      "algo_oid": "N/A"
    },
    "DES-EDE3": {
      "algo_oid": "N/A"
    },
    "DES-EDE3-ECB": {
      "algo_oid": "N/A"
    }
  },
  "ciphers": {
    "tls": {
      "TLS_AES_256_GCM_SHA384": {
        "TLS_version": "TLSv1.3",
        "key_exchange_algorithm": "any",
        "authentication": "any",
        "encryption_algorithm": "AESGCM(256)",
        "mac": "AEAD"
      },
      "TLS_CHACHA20_POLY1305_SHA256": {
        "TLS_version": "TLSv1.3",
        "key_exchange_algorithm": "any",
        "authentication": "any",
        "encryption_algorithm": "CHACHA20/POLY1305(256)",
        "mac": "AEAD"
      },
      "TLS_AES_128_GCM_SHA256": {
        "TLS_version": "TLSv1.3",
        "key_exchange_algorithm": "any",
        "authentication": "any",
        "encryption_algorithm": "AESGCM(128)",
        "mac": "AEAD"
      },
      "ECDHE-ECDSA-AES256-GCM-SHA384": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "ECDH",
        "authentication": "ECDSA",
        "encryption_algorithm": "AESGCM(256)",
        "mac": "AEAD"
      },
      "ECDHE-RSA-AES256-GCM-SHA384": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "ECDH",
        "authentication": "RSA",
        "encryption_algorithm": "AESGCM(256)",
        "mac": "AEAD"
      },
      "DHE-RSA-AES256-GCM-SHA384": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "DH",
        "authentication": "RSA",
        "encryption_algorithm": "AESGCM(256)",
        "mac": "AEAD"
      },
      "ECDHE-ECDSA-CHACHA20-POLY1305": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "ECDH",
        "authentication": "ECDSA",
        "encryption_algorithm": "CHACHA20/POLY1305(256)",
        "mac": "AEAD"
      },
      "ECDHE-RSA-CHACHA20-POLY1305": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "ECDH",
        "authentication": "RSA",
        "encryption_algorithm": "CHACHA20/POLY1305(256)",
        "mac": "AEAD"
      },
      "DHE-RSA-CHACHA20-POLY1305": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "DH",
        "authentication": "RSA",
        "encryption_algorithm": "CHACHA20/POLY1305(256)",
        "mac": "AEAD"
      },
      "ECDHE-ECDSA-AES128-GCM-SHA256": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "ECDH",
        "authentication": "ECDSA",
        "encryption_algorithm": "AESGCM(128)",
        "mac": "AEAD"
      },
      "ECDHE-RSA-AES128-GCM-SHA256": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "ECDH",
        "authentication": "RSA",
        "encryption_algorithm": "AESGCM(128)",
        "mac": "AEAD"
      },
      "DHE-RSA-AES128-GCM-SHA256": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "DH",
        "authentication": "RSA",
        "encryption_algorithm": "AESGCM(128)",
        "mac": "AEAD"
      },
      "ECDHE-ECDSA-AES256-SHA384": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "ECDH",
        "authentication": "ECDSA",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA384"
      },
      "ECDHE-RSA-AES256-SHA384": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "ECDH",
        "authentication": "RSA",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA384"
      },
      "DHE-RSA-AES256-SHA256": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "DH",
        "authentication": "RSA",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA256"
      },
      "ECDHE-ECDSA-AES128-SHA256": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "ECDH",
        "authentication": "ECDSA",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA256"
      },
      "ECDHE-RSA-AES128-SHA256": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "ECDH",
        "authentication": "RSA",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA256"
      },
      "DHE-RSA-AES128-SHA256": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "DH",
        "authentication": "RSA",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA256"
      },
      "ECDHE-ECDSA-AES256-SHA": {
        "TLS_version": "TLSv1",
        "key_exchange_algorithm": "ECDH",
        "authentication": "ECDSA",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA1"
      },
      "ECDHE-RSA-AES256-SHA": {
        "TLS_version": "TLSv1",
        "key_exchange_algorithm": "ECDH",
        "authentication": "RSA",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA1"
      },
      "DHE-RSA-AES256-SHA": {
        "TLS_version": "SSLv3",
        "key_exchange_algorithm": "DH",
        "authentication": "RSA",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA1"
      },
      "ECDHE-ECDSA-AES128-SHA": {
        "TLS_version": "TLSv1",
        "key_exchange_algorithm": "ECDH",
        "authentication": "ECDSA",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA1"
      },
      "ECDHE-RSA-AES128-SHA": {
        "TLS_version": "TLSv1",
        "key_exchange_algorithm": "ECDH",
        "authentication": "RSA",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA1"
      },
      "DHE-RSA-AES128-SHA": {
        "TLS_version": "SSLv3",
        "key_exchange_algorithm": "DH",
        "authentication": "RSA",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA1"
      },
      "RSA-PSK-AES256-GCM-SHA384": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "RSAPSK",
        "authentication": "RSA",
        "encryption_algorithm": "AESGCM(256)",
        "mac": "AEAD"
      },
      "DHE-PSK-AES256-GCM-SHA384": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "DHEPSK",
        "authentication": "PSK",
        "encryption_algorithm": "AESGCM(256)",
        "mac": "AEAD"
      },
      "RSA-PSK-CHACHA20-POLY1305": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "RSAPSK",
        "authentication": "RSA",
        "encryption_algorithm": "CHACHA20/POLY1305(256)",
        "mac": "AEAD"
      },
      "DHE-PSK-CHACHA20-POLY1305": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "DHEPSK",
        "authentication": "PSK",
        "encryption_algorithm": "CHACHA20/POLY1305(256)",
        "mac": "AEAD"
      },
      "ECDHE-PSK-CHACHA20-POLY1305": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "ECDHEPSK",
        "authentication": "PSK",
        "encryption_algorithm": "CHACHA20/POLY1305(256)",
        "mac": "AEAD"
      },
      "AES256-GCM-SHA384": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "RSA",
        "authentication": "RSA",
        "encryption_algorithm": "AESGCM(256)",
        "mac": "AEAD"
      },
      "PSK-AES256-GCM-SHA384": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "PSK",
        "authentication": "PSK",
        "encryption_algorithm": "AESGCM(256)",
        "mac": "AEAD"
      },
      "PSK-CHACHA20-POLY1305": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "PSK",
        "authentication": "PSK",
        "encryption_algorithm": "CHACHA20/POLY1305(256)",
        "mac": "AEAD"
      },
      "RSA-PSK-AES128-GCM-SHA256": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "RSAPSK",
        "authentication": "RSA",
        "encryption_algorithm": "AESGCM(128)",
        "mac": "AEAD"
      },
      "DHE-PSK-AES128-GCM-SHA256": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "DHEPSK",
        "authentication": "PSK",
        "encryption_algorithm": "AESGCM(128)",
        "mac": "AEAD"
      },
      "AES128-GCM-SHA256": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "RSA",
        "authentication": "RSA",
        "encryption_algorithm": "AESGCM(128)",
        "mac": "AEAD"
      },
      "PSK-AES128-GCM-SHA256": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "PSK",
        "authentication": "PSK",
        "encryption_algorithm": "AESGCM(128)",
        "mac": "AEAD"
      },
      "AES256-SHA256": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "RSA",
        "authentication": "RSA",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA256"
      },
      "AES128-SHA256": {
        "TLS_version": "TLSv1.2",
        "key_exchange_algorithm": "RSA",
        "authentication": "RSA",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA256"
      },
      "ECDHE-PSK-AES256-CBC-SHA384": {
        "TLS_version": "TLSv1",
        "key_exchange_algorithm": "ECDHEPSK",
        "authentication": "PSK",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA384"
      },
      "ECDHE-PSK-AES256-CBC-SHA": {
        "TLS_version": "TLSv1",
        "key_exchange_algorithm": "ECDHEPSK",
        "authentication": "PSK",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA1"
      },
      "SRP-RSA-AES-256-CBC-SHA": {
        "TLS_version": "SSLv3",
        "key_exchange_algorithm": "SRP",
        "authentication": "RSA",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA1"
      },
      "SRP-AES-256-CBC-SHA": {
        "TLS_version": "SSLv3",
        "key_exchange_algorithm": "SRP",
        "authentication": "SRP",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA1"
      },
      "RSA-PSK-AES256-CBC-SHA384": {
        "TLS_version": "TLSv1",
        "key_exchange_algorithm": "RSAPSK",
        "authentication": "RSA",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA384"
      },
      "DHE-PSK-AES256-CBC-SHA384": {
        "TLS_version": "TLSv1",
        "key_exchange_algorithm": "DHEPSK",
        "authentication": "PSK",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA384"
      },
      "RSA-PSK-AES256-CBC-SHA": {
        "TLS_version": "SSLv3",
        "key_exchange_algorithm": "RSAPSK",
        "authentication": "RSA",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA1"
      },
      "DHE-PSK-AES256-CBC-SHA": {
        "TLS_version": "SSLv3",
        "key_exchange_algorithm": "DHEPSK",
        "authentication": "PSK",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA1"
      },
      "AES256-SHA": {
        "TLS_version": "SSLv3",
        "key_exchange_algorithm": "RSA",
        "authentication": "RSA",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA1"
      },
      "PSK-AES256-CBC-SHA384": {
        "TLS_version": "TLSv1",
        "key_exchange_algorithm": "PSK",
        "authentication": "PSK",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA384"
      },
      "PSK-AES256-CBC-SHA": {
        "TLS_version": "SSLv3",
        "key_exchange_algorithm": "PSK",
        "authentication": "PSK",
        "encryption_algorithm": "AES(256)",
        "mac": "SHA1"
      },
      "ECDHE-PSK-AES128-CBC-SHA256": {
        "TLS_version": "TLSv1",
        "key_exchange_algorithm": "ECDHEPSK",
        "authentication": "PSK",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA256"
      },
      "ECDHE-PSK-AES128-CBC-SHA": {
        "TLS_version": "TLSv1",
        "key_exchange_algorithm": "ECDHEPSK",
        "authentication": "PSK",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA1"
      },
      "SRP-RSA-AES-128-CBC-SHA": {
        "TLS_version": "SSLv3",
        "key_exchange_algorithm": "SRP",
        "authentication": "RSA",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA1"
      },
      "SRP-AES-128-CBC-SHA": {
        "TLS_version": "SSLv3",
        "key_exchange_algorithm": "SRP",
        "authentication": "SRP",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA1"
      },
      "RSA-PSK-AES128-CBC-SHA256": {
        "TLS_version": "TLSv1",
        "key_exchange_algorithm": "RSAPSK",
        "authentication": "RSA",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA256"
      },
      "DHE-PSK-AES128-CBC-SHA256": {
        "TLS_version": "TLSv1",
        "key_exchange_algorithm": "DHEPSK",
        "authentication": "PSK",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA256"
      },
      "RSA-PSK-AES128-CBC-SHA": {
        "TLS_version": "SSLv3",
        "key_exchange_algorithm": "RSAPSK",
        "authentication": "RSA",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA1"
      },
      "DHE-PSK-AES128-CBC-SHA": {
        "TLS_version": "SSLv3",
        "key_exchange_algorithm": "DHEPSK",
        "authentication": "PSK",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA1"
      },
      "AES128-SHA": {
        "TLS_version": "SSLv3",
        "key_exchange_algorithm": "RSA",
        "authentication": "RSA",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA1"
      },
      "PSK-AES128-CBC-SHA256": {
        "TLS_version": "TLSv1",
        "key_exchange_algorithm": "PSK",
        "authentication": "PSK",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA256"
      },
      "PSK-AES128-CBC-SHA": {
        "TLS_version": "SSLv3",
        "key_exchange_algorithm": "PSK",
        "authentication": "PSK",
        "encryption_algorithm": "AES(128)",
        "mac": "SHA1"
      }
    }
  },
  "nmap_tls_info": {
    "nmap_tls_ssl_enum_ciphers_info": {
      "22": {
        "net_proto": "tcp",
        "status": "open",
        "tls_vers_enabled": {}
      }
    }
  },
  "certificate": {},
  "ssh_crypto_info": {
    "gssapikexalgorithms": [
      "gss-group14-sha256-",
      "gss-group16-sha512-",
      "gss-nistp256-sha256-",
      "gss-curve25519-sha256-",
      "gss-group14-sha1-",
      "gss-gex-sha1-"
    ],
    "ciphers": [
      "chacha20-poly1305@openssh.com",
      "aes128-ctr",
      "aes192-ctr",
      "aes256-ctr",
      "aes128-gcm@openssh.com",
      "aes256-gcm@openssh.com"
    ],
    "macs": [
      "umac-64-etm@openssh.com",
      "umac-128-etm@openssh.com",
      "hmac-sha2-256-etm@openssh.com",
      "hmac-sha2-512-etm@openssh.com",
      "hmac-sha1-etm@openssh.com",
      "umac-64@openssh.com",
      "umac-128@openssh.com",
      "hmac-sha2-256",
      "hmac-sha2-512",
      "hmac-sha1"
    ],
    "kexalgorithms": [
      "sntrup761x25519-sha512@openssh.com",
      "curve25519-sha256",
      "curve25519-sha256@libssh.org",
      "ecdh-sha2-nistp256",
      "ecdh-sha2-nistp384",
      "ecdh-sha2-nistp521",
      "diffie-hellman-group-exchange-sha256",
      "diffie-hellman-group16-sha512",
      "diffie-hellman-group18-sha512",
      "diffie-hellman-group14-sha256"
    ],
    "casignaturealgorithms": [
      "ssh-ed25519",
      "ecdsa-sha2-nistp256",
      "ecdsa-sha2-nistp384",
      "ecdsa-sha2-nistp521",
      "sk-ssh-ed25519@openssh.com",
      "sk-ecdsa-sha2-nistp256@openssh.com",
      "rsa-sha2-512",
      "rsa-sha2-256"
    ],
    "hostbasedacceptedalgorithms": [
      "ssh-ed25519-cert-v01@openssh.com",
      "ecdsa-sha2-nistp256-cert-v01@openssh.com",
      "ecdsa-sha2-nistp384-cert-v01@openssh.com",
      "ecdsa-sha2-nistp521-cert-v01@openssh.com",
      "sk-ssh-ed25519-cert-v01@openssh.com",
      "sk-ecdsa-sha2-nistp256-cert-v01@openssh.com",
      "rsa-sha2-512-cert-v01@openssh.com",
      "rsa-sha2-256-cert-v01@openssh.com",
      "ssh-ed25519",
      "ecdsa-sha2-nistp256",
      "ecdsa-sha2-nistp384",
      "ecdsa-sha2-nistp521",
      "sk-ssh-ed25519@openssh.com",
      "sk-ecdsa-sha2-nistp256@openssh.com",
      "rsa-sha2-512",
      "rsa-sha2-256"
    ],
    "hostkeyalgorithms": [
      "ssh-ed25519-cert-v01@openssh.com",
      "ecdsa-sha2-nistp256-cert-v01@openssh.com",
      "ecdsa-sha2-nistp384-cert-v01@openssh.com",
      "ecdsa-sha2-nistp521-cert-v01@openssh.com",
      "sk-ssh-ed25519-cert-v01@openssh.com",
      "sk-ecdsa-sha2-nistp256-cert-v01@openssh.com",
      "rsa-sha2-512-cert-v01@openssh.com",
      "rsa-sha2-256-cert-v01@openssh.com",
      "ssh-ed25519",
      "ecdsa-sha2-nistp256",
      "ecdsa-sha2-nistp384",
      "ecdsa-sha2-nistp521",
      "sk-ssh-ed25519@openssh.com",
      "sk-ecdsa-sha2-nistp256@openssh.com",
      "rsa-sha2-512",
      "rsa-sha2-256"
    ],
    "pubkeyacceptedalgorithms": [
      "ssh-ed25519-cert-v01@openssh.com",
      "ecdsa-sha2-nistp256-cert-v01@openssh.com",
      "ecdsa-sha2-nistp384-cert-v01@openssh.com",
      "ecdsa-sha2-nistp521-cert-v01@openssh.com",
      "sk-ssh-ed25519-cert-v01@openssh.com",
      "sk-ecdsa-sha2-nistp256-cert-v01@openssh.com",
      "rsa-sha2-512-cert-v01@openssh.com",
      "rsa-sha2-256-cert-v01@openssh.com",
      "ssh-ed25519",
      "ecdsa-sha2-nistp256",
      "ecdsa-sha2-nistp384",
      "ecdsa-sha2-nistp521",
      "sk-ssh-ed25519@openssh.com",
      "sk-ecdsa-sha2-nistp256@openssh.com",
      "rsa-sha2-512",
      "rsa-sha2-256"
    ]
  }
}

# # Test with:
# print("\nValidation of CycloneDX 1.4 payload with function validate_cyclonedx_bom:")
# # Validate the payload using the CycloneDX Python Library
# # and the custom validation function.
# # Note: The validate_cyclonedx_bom function is fully implemented.
# # It uses the JsonValidator for standard validation.
# # It uses the JsonStrictValidator for strict validation.
# result = validate_cyclonedx_bom(payload_correct_v14, strict=False)
# print(result)

# print("\nValidation AD of CycloneDX 1.4 payload with function validate_cyclone:")
# # Validate the payload using the CycloneDX Python Library
# # and the custom validation function.
# # Note: The validate_cyclone function is not fully implemented in this example. 
# result2 = validate_cyclone(payload_correct_v14)
# print(result2)

# print("\nSmart validation of CycloneDX 1.4 payload:")
# result_smart = validate_cyclonedx_smart(payload_correct_v14)
# print(result_smart)

print("\nQuick validation of CycloneDX from Master 1 example payload:")
result_quick = validate_cyclonedx_smart(payload_correct_bom_master1, strict=True, thorough=True)
print(result_quick)

print("\nQuick validation of CycloneDX from Master 2 example payload:")
result_quick = validate_cyclonedx_smart(payload_correct_bom_master2, strict=True, thorough=True)
print(result_quick)

print("\nQuick validation of CycloneDX from Master 3 example payload:")
result_quick = validate_cyclonedx_smart(payload_correct_bom_master3, strict=True, thorough=True)
print(result_quick)

print("\nQuick validation of CycloneDX from CCM1 example payload:")
result_quick = validate_cyclonedx_smart(payload_correct_CCM1, strict=True, thorough=True)
print(result_quick)

print("\nQuick validation of CycloneDX from CCM2 example payload:")
result_quick = validate_cyclonedx_smart(payload_correct_CCM2, strict=True, thorough=True)
print(result_quick)

print("\nQuick validation of CycloneDX from CCM3 example payload:")
result_quick = validate_cyclonedx_smart(payload_correct_CCM3, strict=True, thorough=True)
print(result_quick)

print("\nQuick validation of CycloneDX from CCM4 example payload:")
result_quick = validate_cyclonedx_smart(payload_correct_CCM4, strict=True, thorough=True)
print(result_quick)

print("\nQuick validation of CycloneDX from CCM5 example payload:")
result_quick = validate_cyclonedx_smart(payload_correct_CCM5, strict=True, thorough=True)
print(result_quick)


Quick validation of CycloneDX from Master 1 example payload:
{'success': True, 'message': 'Validation passed. Schema version: 1.6', 'schema_version': '1.6'}

Quick validation of CycloneDX from Master 2 example payload:
{'success': True, 'message': 'Validation passed. Schema version: 1.4', 'schema_version': '1.4'}

Quick validation of CycloneDX from Master 3 example payload:
{'success': True, 'message': 'Validation passed. Schema version: 1.4', 'schema_version': '1.4'}

Quick validation of CycloneDX from CCM1 example payload:
{'success': False, 'message': 'Validation failed with errors: \'certificate\' is not one of [\'application\', \'framework\', \'library\', \'container\', \'platform\', \'operating-system\', \'device\', \'device-driver\', \'firmware\', \'file\', \'machine-learning-model\', \'data\', \'cryptographic-asset\']\n\nFailed validating \'enum\' in schema[\'properties\'][\'components\'][\'items\'][\'properties\'][\'type\']:\n    {\'type\': \'string\',\n     \'enum\': [\'appl

In [ ]:
import unittest
import json
import sys
import os
import logging
from unittest.mock import patch, MagicMock


from .adapter import DTModelAdapter 

class TestFromCyclone(unittest.TestCase):
    def setUp(self):
        self.adapter = DTModelAdapter()
        # Set up logging
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)
        
        # Sample CycloneDX BOM with all possible sections
        self.complete_cyclone_json = json.dumps({
            "bomFormat": "CycloneDX",
            "specVersion": "1.6",
            "serialNumber": "urn:uuid:3e671687-395b-41f5-a30f-a58921a69b79",
            "version": 1,
            "metadata": {
                "timestamp": "2024-04-02T10:30:00Z",
                "tools": [
                    {"name": "test-tool", "version": "1.0.0"}
                ],
                "component": {
                    "type": "application",
                    "name": "test-app",
                    "version": "1.2.3"
                }
            },
            "components": [
                {
                    "type": "library",
                    "name": "component1",
                    "version": "1.0.0"
                }
            ],
            "services": [
                {
                    "name": "service1",
                    "endpoint": "https://example.com/api"
                }
            ],
            "dependencies": [
                {
                    "ref": "component1",
                    "dependsOn": []
                }
            ],
            "compositions": [
                {
                    "aggregate": "complete",
                    "assemblies": ["component1"]
                }
            ],
            "vulnerabilities": [
                {
                    "id": "CVE-2023-12345",
                    "source": {"name": "NVD"},
                    "ratings": [{"severity": "high"}]
                }
            ],
            "formulations": [
                {
                    "components": ["component1"]
                }
            ],
            "annotations": [
                {
                    "subjects": ["component1"],
                    "text": "Test annotation"
                }
            ],
            "definitions": [
                {
                    "id": "def1",
                    "name": "test definition"
                }
            ],
            "declarations": [
                {
                    "id": "decl1",
                    "name": "test declaration"
                }
            ],
            "extensions": [
                {
                    "id": "ext1",
                    "name": "test extension"
                }
            ],
            "extraAttribute": "test"
        })
        
        # Simple CycloneDX BOM with only essential fields
        self.simple_cyclone_json = json.dumps({
            "bomFormat": "CycloneDX",
            "specVersion": "1.6",
            "serialNumber": "urn:uuid:3e671687-395b-41f5-a30f-a58921a69b79",
            "version": 1,
            "metadata": {
                "timestamp": "2024-04-02T10:30:00Z"
            },
            "components": [
                {
                    "type": "library",
                    "name": "component1",
                    "version": "1.0.0"
                }
            ]
        })

    def test_from_cyclone_complete(self):
        """Test from_cyclone with a complete CycloneDX payload containing all possible sections"""
        result = self.adapter.from_cyclone(self.complete_cyclone_json, "test_name")
        
        # Verify the structure of the result
        self.assertIsInstance(result, dict)
        self.assertIn("attributes", result)
        self.assertIn("features", result)
        self.assertIn("_metadata", result)
        
        # Verify attributes contain the expected top-level fields
        self.assertEqual(result["attributes"]["bomFormat"], "CycloneDX")
        self.assertEqual(result["attributes"]["specVersion"], "1.6")
        self.assertEqual(result["attributes"]["serialNumber"], "urn:uuid:3e671687-395b-41f5-a30f-a58921a69b79")
        self.assertEqual(result["attributes"]["extraAttribute"], "test")
        
        # Verify features contain all the expected sections
        self.assertIn("metadata", result["features"])
        self.assertIn("components", result["features"])
        self.assertIn("services", result["features"])
        self.assertIn("dependencies", result["features"])
        self.assertIn("compositions", result["features"])
        self.assertIn("vulnerabilities", result["features"])
        self.assertIn("formulations", result["features"])
        self.assertIn("annotations", result["features"])
        self.assertIn("definitions", result["features"])
        self.assertIn("declarations", result["features"])
        self.assertIn("extensions", result["features"])
        
        # Check that component properties are correctly indexed
        self.assertIn("1", result["features"]["components"]["properties"])
        self.assertEqual(result["features"]["components"]["properties"]["1"]["name"], "component1")
        
        # Verify _metadata contains the full original payload
        self.assertEqual(result["_metadata"], json.loads(self.complete_cyclone_json))

    def test_from_cyclone_simple(self):
        """Test from_cyclone with a simple CycloneDX payload containing only essential fields"""
        result = self.adapter.from_cyclone(self.simple_cyclone_json, "test_name")
        
        # Verify the structure of the result
        self.assertIsInstance(result, dict)
        self.assertIn("attributes", result)
        self.assertIn("features", result)
        
        # Verify attributes contain the expected top-level fields
        self.assertEqual(result["attributes"]["bomFormat"], "CycloneDX")
        self.assertEqual(result["attributes"]["specVersion"], "1.6")
        
        # Verify only the sections from the input are in features
        self.assertIn("metadata", result["features"])
        self.assertIn("components", result["features"])
        self.assertNotIn("services", result["features"])  # Not in the input
        
        # Check that metadata properties are correctly preserved
        self.assertEqual(
            result["features"]["metadata"]["properties"]["timestamp"], 
            "2024-04-02T10:30:00Z"
        )

    def test_from_cyclone_missing_optional_sections(self):
        """Test from_cyclone with a CycloneDX payload missing optional sections"""
        # Create JSON without metadata
        no_metadata_json = json.dumps({
            "bomFormat": "CycloneDX",
            "specVersion": "1.6",
            "serialNumber": "urn:uuid:test",
            "components": []
        })
        
        result = self.adapter.from_cyclone(no_metadata_json, "test_name")
        
        # Should still have attributes and features
        self.assertIn("attributes", result)
        self.assertIn("features", result)
        
        # Should not have metadata in features
        self.assertNotIn("metadata", result["features"])
        
        # Components should be an empty dict
        self.assertIn("components", result["features"])
        self.assertEqual(result["features"]["components"]["properties"], {})

    @patch('logging.Logger.info')
    def test_logging(self, mock_logger_info):
        """Test that the function logs correctly at the beginning and end"""
        self.adapter.from_cyclone(self.simple_cyclone_json, "test_name")
        
        # Check that the logger.info was called with the expected messages
        mock_logger_info.assert_any_call("Starting transformation of CycloneDX payload")
        mock_logger_info.assert_any_call("Successfully transformed CycloneDX payload")

    def test_from_cyclone_invalid_json(self):
        """Test from_cyclone with invalid JSON input"""
        invalid_json = "not a valid json"
        
        with self.assertRaises(json.JSONDecodeError):
            self.adapter.from_cyclone(invalid_json, "test_name")

if __name__ == '__main__':
    unittest.main()

ModuleNotFoundError: No module named 'core'

In [5]:
import random
from datetime import datetime


print(str(''.join(filter(str.isdigit, datetime.now().isoformat())))+"-"+str(random.randint(1000000000, 9999999999)))


20250407150444851514-3157900109
